# anim-gen — training notebook (Kaggle & Colab)

Train an image generator on a **free Kaggle or Colab GPU**: a GAN (DCGAN or
ResNet-SNGAN) or a diffusion model (DDPM/DDIM U-Net), on MNIST, CIFAR-10 or
**AFHQ animal faces** (cat / dog / wild).

The notebook is **self-contained**: section 2 writes the `anim_gen` package
(identical to the repo) to disk, then everything uses it. Pick a preset in
section 3 and press **Run all**.

| | Kaggle | Colab |
|---|---|---|
| GPU | *Settings → Accelerator → GPU T4 x2* (or P100) | *Runtime → Change runtime type → T4 GPU* |
| Internet | *Settings → Internet → On* (to download data) | on by default |
| Session limit | 12 h per session, 30 h GPU / week | ~12 h, can disconnect when idle |
| Checkpoints kept in | `/kaggle/working` (saved with "Save Version") | Google Drive if `USE_GOOGLE_DRIVE = True` |

**Sections:** 1. Install · 2. The package · 3. Settings · 4. Data · 5. Model ·
6. Live dashboard · 7. Training · 8. Results · 9. Download

## 1. Install

Kaggle and Colab already have PyTorch, torchvision and matplotlib. `datasets`
is only needed to download AFHQ from the Hugging Face Hub.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets>=3.6"])
print("ready")

## 2. The `anim_gen` package

Each cell below writes one file of the repo's `src/anim_gen/` package
(generated from the repo by `notebooks/build_notebook.py`). You do not need to
read them to train; they are here so the notebook runs anywhere, offline.

In [ ]:
import os

os.makedirs("anim_gen/models", exist_ok=True)

`anim_gen/config.py`: every hyperparameter + the named presets

In [ ]:
%%writefile anim_gen/config.py
"""All hyperparameters live here, grouped by what they control.

Configs are frozen dataclasses: they cannot be changed by accident after
creation, and they are saved inside every checkpoint so a run can be
reproduced and a checkpoint can rebuild its own model.

PRESETS at the bottom are named starting points (e.g. "cifar10-resnet").
On the command line every field can still be overridden one by one:
    anim-train --preset cifar10-resnet --batch-size 32 --steps 2000
"""

from dataclasses import asdict, dataclass, fields
from pathlib import Path
from typing import Any

DATASETS = ("mnist", "fashion-mnist", "cifar10", "folder", "shapes")
GRAYSCALE_DATASETS = ("mnist", "fashion-mnist")
ARCHS = ("dcgan", "resnet", "unet")  # dcgan / resnet are GANs, unet is a diffusion model
GAN_LOSSES = ("hinge", "nonsaturating", "wgan-gp", "lsgan")
PRECISIONS = ("auto", "fp32", "bf16")


@dataclass(frozen=True, slots=True)
class DataConfig:
    """Which images we learn from and how they become tensors in [-1, 1]."""

    dataset: str = "cifar10"
    data_dir: Path = Path("data")  # download target, or the image folder for "folder"
    image_size: int = 32
    hflip: bool = True  # random horizontal flips (never for digits: a flipped 3 is not a 3)
    conditional: bool = False  # learn p(image | class) from the dataset labels
    limit: int | None = None  # use only the first N images (tiny overfit runs)
    download: bool = True  # torchvision datasets only

    def __post_init__(self) -> None:
        if self.dataset not in DATASETS:
            raise ValueError(f"dataset must be one of {DATASETS}")
        if self.image_size < 8 or self.image_size & (self.image_size - 1):
            raise ValueError("image_size must be a power of two >= 8 (8, 16, 32, 64, ...)")

    @property
    def channels(self) -> int:
        """1 for grayscale datasets, 3 (RGB) for everything else."""
        return 1 if self.dataset in GRAYSCALE_DATASETS else 3


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Shape of the networks. image_size / channels / num_classes come from the data."""

    arch: str = "resnet"
    image_size: int = 32
    channels: int = 3
    num_classes: int = 0  # 0 = unconditional

    # --- GANs (dcgan, resnet) ---
    z_dim: int = 128  # size of the random noise vector the generator starts from
    g_channels: int = 64  # generator width at full resolution; doubles at each lower one
    d_channels: int = 64  # discriminator width at full resolution
    max_channels: int = 512  # cap on the doubling
    attn_resolution: int = 0  # self-attention block at this resolution (0 = none)
    spectral_norm: bool = True  # spectral norm on every discriminator layer

    # --- diffusion (unet) ---
    unet_channels: int = 64  # width of the first U-Net level
    channel_mult: tuple[int, ...] = (1, 2, 2, 2)  # width multiplier per level
    num_res_blocks: int = 2  # residual blocks per level
    attn_resolutions: tuple[int, ...] = (16,)  # resolutions that get self-attention
    dropout: float = 0.1
    timesteps: int = 1000  # length of the noising process
    noise_schedule: str = "cosine"  # "cosine" | "linear"
    prediction: str = "eps"  # what the U-Net predicts: noise ("eps") or velocity ("v")
    cfg_dropout: float = 0.1  # P(drop the label) in training -> classifier-free guidance

    def __post_init__(self) -> None:
        if self.arch not in ARCHS:
            raise ValueError(f"arch must be one of {ARCHS}")
        if self.image_size < 8 or self.image_size & (self.image_size - 1):
            raise ValueError("image_size must be a power of two >= 8")
        if self.noise_schedule not in ("cosine", "linear"):
            raise ValueError("noise_schedule must be cosine or linear")
        if self.prediction not in ("eps", "v"):
            raise ValueError("prediction must be eps or v")
        if not 0.0 <= self.dropout < 1.0 or not 0.0 <= self.cfg_dropout < 1.0:
            raise ValueError("dropout and cfg_dropout must be in [0, 1)")
        if self.image_size >> (len(self.channel_mult) - 1) < 4 and self.arch == "unet":
            raise ValueError("too many U-Net levels for this image_size (lowest must be >= 4)")

    @property
    def family(self) -> str:
        """ "gan" or "diffusion": decides which training task is used."""
        return "diffusion" if self.arch == "unet" else "gan"


@dataclass(frozen=True, slots=True)
class TrainConfig:
    """Optimisation and bookkeeping settings for one training run."""

    output_dir: Path = Path("checkpoints/run")
    seed: int = 42
    steps: int = 50_000  # generator (GAN) or denoiser (diffusion) optimizer steps
    batch_size: int = 64  # per process (per GPU)

    # --- GANs ---
    loss: str = "hinge"
    lr_g: float = 2e-4
    lr_d: float = 2e-4
    beta1: float = 0.0  # Adam betas; GANs like a small beta1 (0.0 or 0.5)
    beta2: float = 0.9
    d_steps: int = 1  # discriminator updates per generator update (5 for wgan-gp)
    gp_weight: float = 10.0  # gradient penalty weight (wgan-gp only)
    r1_gamma: float = 0.0  # R1 penalty on real images (0 = off)
    diffaug: str = ""  # DiffAugment policy, e.g. "color,translation,cutout" (small datasets)

    # --- diffusion ---
    lr: float = 2e-4
    warmup_steps: int = 1000
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    sample_steps: int = 100  # DDIM steps used for preview grids and FID
    guidance_scale: float = 1.0  # classifier-free guidance for previews (1 = off)

    # --- both ---
    ema_decay: float = 0.999  # exponential moving average of the weights (0 = off)
    ema_start: int = 0  # step from which the EMA starts tracking
    precision: str = "auto"  # "auto" (bf16 on Ampere+ GPUs, else fp32) | "fp32" | "bf16"
    num_workers: int = 4
    log_every: int = 100  # steps between log lines
    sample_every: int = 1000  # steps between preview grids (samples/step_XXXXXX.png)
    ckpt_every: int = 5000  # steps between checkpoints
    fid_every: int = 0  # steps between FID evaluations (0 = off; needs Inception weights)
    fid_samples: int = 5000
    num_samples: int = 64  # images in each preview grid
    device: str = "auto"  # "auto" | "cpu" | "cuda" | "mps"
    resume: Path | None = None  # checkpoint to continue from
    tensorboard: bool = False  # also log to TensorBoard (needs the tensorboard package)
    time_budget_hours: float = 0.0  # save and stop after this long (0 = no limit), e.g. Kaggle

    def __post_init__(self) -> None:
        if self.steps <= 0 or self.batch_size <= 0 or self.d_steps <= 0:
            raise ValueError("steps, batch_size and d_steps must be positive")
        if self.loss not in GAN_LOSSES:
            raise ValueError(f"loss must be one of {GAN_LOSSES}")
        if self.precision not in PRECISIONS:
            raise ValueError(f"precision must be one of {PRECISIONS}")
        if not 0.0 <= self.ema_decay < 1.0:
            raise ValueError("ema_decay must be in [0, 1)")


def config_to_dict(config: Any) -> dict[str, Any]:
    """Turn a config into plain types so it can be stored in a checkpoint."""
    out = {}
    for key, value in asdict(config).items():
        if isinstance(value, Path):
            value = str(value)
        elif isinstance(value, tuple):
            value = list(value)
        out[key] = value
    return out


def config_from_dict(cls: type, data: dict[str, Any]) -> Any:
    """Inverse of config_to_dict. Unknown keys (from newer versions) are ignored."""
    kwargs = {}
    for f in fields(cls):
        if f.name not in data:
            continue
        value = data[f.name]
        if isinstance(value, list):
            value = tuple(value)
        elif value is not None and f.name in ("data_dir", "output_dir", "resume"):
            value = Path(value)
        kwargs[f.name] = value
    return cls(**kwargs)


# ---------------------------------------------------------------------------
# Presets: flat {field name: value} overrides of the three configs above.
# Field names are unique across DataConfig / ModelConfig / TrainConfig.
# ---------------------------------------------------------------------------

_DCGAN = {
    "arch": "dcgan", "loss": "nonsaturating", "beta1": 0.5, "beta2": 0.999,
    "spectral_norm": False, "z_dim": 100, "ema_decay": 0.999, "ema_start": 1000,
}  # fmt: skip
_RESNET = {
    "arch": "resnet", "loss": "hinge", "beta1": 0.0, "beta2": 0.9, "spectral_norm": True,
    "z_dim": 128, "ema_decay": 0.9999, "ema_start": 1000,
}  # fmt: skip
_UNET = {"arch": "unet", "lr": 2e-4, "ema_decay": 0.9995, "beta1": 0.9, "beta2": 0.999}

PRESETS: dict[str, dict[str, Any]] = {
    # --- MNIST (1 x 32 x 32): minutes on a GPU, sanity checks on a CPU ---
    "mnist-dcgan": {
        **_DCGAN, "dataset": "mnist", "image_size": 32, "hflip": False,
        "g_channels": 32, "d_channels": 32, "max_channels": 256, "steps": 10_000,
    },
    "mnist-diffusion": {
        **_UNET, "dataset": "mnist", "image_size": 32, "hflip": False, "conditional": True,
        "unet_channels": 32, "channel_mult": (1, 2, 2), "attn_resolutions": (8,),
        "steps": 20_000, "batch_size": 128,
    },
    # --- CIFAR-10 (3 x 32 x 32) ---
    "cifar10-dcgan": {
        **_DCGAN, "dataset": "cifar10", "image_size": 32,
        "g_channels": 64, "d_channels": 64, "max_channels": 512, "steps": 50_000,
    },
    "cifar10-resnet": {
        **_RESNET, "dataset": "cifar10", "image_size": 32, "conditional": True,
        "g_channels": 128, "d_channels": 128, "max_channels": 256, "d_steps": 2,
        "attn_resolution": 16, "steps": 60_000, "lr_g": 2e-4, "lr_d": 2e-4,
    },
    "cifar10-diffusion": {
        **_UNET, "dataset": "cifar10", "image_size": 32, "conditional": True,
        "unet_channels": 128, "channel_mult": (1, 2, 2, 2), "attn_resolutions": (16,),
        "dropout": 0.1, "steps": 200_000, "batch_size": 128,
    },
    # --- AFHQ animal faces (cat / dog / wild), prepared with anim-prepare ---
    "afhq64-resnet": {
        **_RESNET, "dataset": "folder", "data_dir": Path("data/afhq64"), "image_size": 64,
        "conditional": True, "g_channels": 64, "d_channels": 64, "max_channels": 512,
        "attn_resolution": 32, "diffaug": "color,translation,cutout", "steps": 100_000,
    },
    "afhq128-resnet": {
        **_RESNET, "dataset": "folder", "data_dir": Path("data/afhq128"), "image_size": 128,
        "conditional": True, "g_channels": 64, "d_channels": 64, "max_channels": 512,
        "attn_resolution": 64, "diffaug": "color,translation,cutout", "r1_gamma": 0.0,
        "steps": 150_000, "batch_size": 32,
    },
    "afhq64-diffusion": {
        **_UNET, "dataset": "folder", "data_dir": Path("data/afhq64"), "image_size": 64,
        "conditional": True, "unet_channels": 128, "channel_mult": (1, 2, 3, 4),
        "attn_resolutions": (16, 8), "prediction": "v", "steps": 300_000, "batch_size": 64,
    },
    "afhq128-diffusion": {
        **_UNET, "dataset": "folder", "data_dir": Path("data/afhq128"), "image_size": 128,
        "conditional": True, "unet_channels": 128, "channel_mult": (1, 1, 2, 3, 4),
        "attn_resolutions": (16, 8), "prediction": "v", "steps": 400_000, "batch_size": 32,
    },
    # --- procedural shapes: no download, used by the tests and quick checks ---
    "shapes-dcgan": {
        **_DCGAN, "dataset": "shapes", "image_size": 16, "g_channels": 16, "d_channels": 16,
        "max_channels": 64, "z_dim": 32, "steps": 500, "num_workers": 0, "ema_start": 100,
    },
    "shapes-diffusion": {
        **_UNET, "dataset": "shapes", "image_size": 16, "unet_channels": 16,
        "channel_mult": (1, 2), "attn_resolutions": (8,), "timesteps": 200,
        "sample_steps": 20, "steps": 500, "num_workers": 0, "warmup_steps": 50,
        "lr": 1e-3, "ema_decay": 0.99,
    },
}  # fmt: skip


def check_presets() -> None:
    """Every preset key must be a field of exactly one config (catches typos)."""
    owners: dict[str, str] = {}
    for cls in (DataConfig, ModelConfig, TrainConfig):
        for f in fields(cls):
            if f.name in owners and cls is not ModelConfig:
                raise AssertionError(f"field {f.name} defined twice")
            owners.setdefault(f.name, cls.__name__)
    for name, preset in PRESETS.items():
        unknown = set(preset) - set(owners)
        if unknown:
            raise AssertionError(f"preset {name}: unknown fields {sorted(unknown)}")


__all__ = [
    "ARCHS",
    "DATASETS",
    "GAN_LOSSES",
    "PRESETS",
    "DataConfig",
    "ModelConfig",
    "TrainConfig",
    "check_presets",
    "config_from_dict",
    "config_to_dict",
]


`anim_gen/data.py`: datasets: MNIST, CIFAR-10, image folders, procedural shapes

In [ ]:
%%writefile anim_gen/data.py
"""Datasets: every item is (image, label), image a float tensor in [-1, 1].

[-1, 1] matches the generator's tanh output and the diffusion model's
Gaussian noise scale. Supported sources:

- "mnist", "fashion-mnist", "cifar10": torchvision, downloaded on first use
- "folder": any directory of images. Sub-directories become classes
  (data/afhq64/cat/*.png, data/afhq64/dog/*.png, ...); a flat folder is one class.
  Use `anim-prepare` to build one from a Hugging Face dataset such as AFHQ.
- "shapes": procedural circles / squares / triangles, generated on the fly.
  No download: used by the tests and for quick checks of the whole pipeline.
"""

from __future__ import annotations

from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset, Subset
from torchvision import datasets, transforms

from anim_gen.config import DataConfig

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}


def build_transform(config: DataConfig, train: bool = True) -> transforms.Compose:
    """PIL image -> [C, H, W] float tensor in [-1, 1]."""
    steps: list = [
        # Resize the short side, then crop the centre: keeps the aspect ratio.
        transforms.Resize(config.image_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(config.image_size),
    ]
    if train and config.hflip:
        steps.append(transforms.RandomHorizontalFlip())
    steps += [
        transforms.ToTensor(),  # [0, 1]
        transforms.Normalize([0.5] * config.channels, [0.5] * config.channels),  # [-1, 1]
    ]
    return transforms.Compose(steps)


class ImageFolderDataset(Dataset):
    """All images under `root`, recursively. First-level sub-folders are classes."""

    def __init__(self, root: Path, transform=None) -> None:
        root = Path(root)
        if not root.is_dir():
            raise FileNotFoundError(f"image folder not found: {root} (see docs/datasets.md)")
        subdirs = sorted(p.name for p in root.iterdir() if p.is_dir())
        self.classes = subdirs or [root.name]
        self.samples: list[tuple[Path, int]] = []
        for label, name in enumerate(self.classes):
            folder = root / name if subdirs else root
            self.samples += [
                (path, label)
                for path in sorted(folder.rglob("*"))
                if path.suffix.lower() in IMAGE_EXTENSIONS
            ]
        if not self.samples:
            raise FileNotFoundError(f"no images ({', '.join(IMAGE_EXTENSIONS)}) under {root}")
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        path, label = self.samples[index]
        with Image.open(path) as img:
            image = img.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label


class ShapesDataset(Dataset):
    """Procedural images: one coloured shape on a dark background.

    Class 0 = circle, 1 = square, 2 = triangle. Item i is always the same image
    (generated from seed i), so the dataset is deterministic and needs no disk.
    """

    def __init__(self, size: int = 2048, image_size: int = 32, hflip: bool = False) -> None:
        self.classes = ["circle", "square", "triangle"]
        self.size = size
        self.image_size = image_size
        self.hflip = hflip
        coords = (torch.arange(image_size) + 0.5) / image_size  # pixel centres in (0, 1)
        self.yy, self.xx = torch.meshgrid(coords, coords, indexing="ij")

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        g = torch.Generator().manual_seed(index)
        label = index % 3
        cx, cy = (0.3 + 0.4 * torch.rand(2, generator=g)).tolist()
        r = 0.15 + 0.15 * torch.rand(1, generator=g).item()
        dx, dy = self.xx - cx, self.yy - cy
        if label == 0:
            mask = dx**2 + dy**2 <= r**2
        elif label == 1:
            mask = (dx.abs() <= r) & (dy.abs() <= r)
        else:  # upward triangle: below the two slanted edges, above the base
            mask = (dy <= r) & (dy >= -r + 2 * dx.abs())
        colour = 0.4 + 0.6 * torch.rand(3, 1, 1, generator=g)
        background = 0.15 * torch.rand(3, 1, 1, generator=g)
        image = torch.where(mask, colour, background)  # [3, H, W] in [0, 1]
        if self.hflip and torch.rand(()) < 0.5:
            image = image.flip(-1)
        return image * 2 - 1, label


class _Limited(Subset):
    """Subset that keeps the parent's `classes`."""

    def __init__(self, dataset: Dataset, n: int) -> None:
        super().__init__(dataset, range(min(n, len(dataset))))
        self.classes = dataset_classes(dataset)


def dataset_classes(dataset: Dataset) -> list[str]:
    """Class names of a dataset (one unnamed class when it has none)."""
    return list(getattr(dataset, "classes", None) or ["all"])


def build_dataset(config: DataConfig, train: bool = True) -> Dataset:
    """The dataset described by `config`, split "train" or "test"."""
    transform = build_transform(config, train=train)
    root = Path(config.data_dir)
    if config.dataset == "mnist":
        dataset = datasets.MNIST(root, train=train, transform=transform, download=config.download)
    elif config.dataset == "fashion-mnist":
        dataset = datasets.FashionMNIST(
            root, train=train, transform=transform, download=config.download
        )
    elif config.dataset == "cifar10":
        dataset = datasets.CIFAR10(root, train=train, transform=transform, download=config.download)
    elif config.dataset == "folder":
        dataset = ImageFolderDataset(root, transform=transform)  # no split: generative models
    elif config.dataset == "shapes":
        dataset = ShapesDataset(2048 if train else 512, config.image_size, config.hflip and train)
    else:  # pragma: no cover - DataConfig already validates the name
        raise ValueError(config.dataset)
    if config.limit is not None:
        dataset = _Limited(dataset, config.limit)
    return dataset


def num_classes(config: DataConfig, dataset: Dataset) -> int:
    """Number of classes the model is conditioned on (0 = unconditional)."""
    return len(dataset_classes(dataset)) if config.conditional else 0


__all__ = [
    "ImageFolderDataset",
    "ShapesDataset",
    "build_dataset",
    "build_transform",
    "dataset_classes",
    "num_classes",
]


`anim_gen/images.py`: tensor -> PNG helpers

In [ ]:
%%writefile anim_gen/images.py
"""Turn model outputs (tensors in [-1, 1]) into image files."""

from __future__ import annotations

import math
from pathlib import Path

import torch
from PIL import Image
from torchvision.utils import make_grid


def to_uint8(images: torch.Tensor) -> torch.Tensor:
    """[-1, 1] float -> [0, 255] uint8, same shape."""
    return ((images.float().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)


def image_grid(images: torch.Tensor, nrow: int | None = None, padding: int = 2) -> torch.Tensor:
    """[N, C, H, W] in [-1, 1] -> one [3, H', W'] uint8 grid, as square as possible."""
    nrow = nrow or max(1, int(math.sqrt(images.shape[0])))
    images = images.float().clamp(-1, 1)
    if images.shape[1] == 1:
        images = images.expand(-1, 3, -1, -1)  # grayscale -> RGB so every grid saves the same
    grid = make_grid(images, nrow=nrow, padding=padding, pad_value=-1.0)
    return to_uint8(grid)


def save_grid(images: torch.Tensor, path: Path, nrow: int | None = None) -> Path:
    """Save a batch as one PNG grid. Returns the path."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    grid = image_grid(images.detach().cpu(), nrow=nrow)
    Image.fromarray(grid.permute(1, 2, 0).numpy()).save(path)
    return path


def save_images(images: torch.Tensor, folder: Path, start: int = 0) -> list[Path]:
    """Save every image of a batch as its own PNG: folder/000000.png, ..."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, image in enumerate(to_uint8(images.detach().cpu())):
        array = image.permute(1, 2, 0).numpy()
        if array.shape[2] == 1:
            array = array[:, :, 0]  # PIL wants [H, W] for grayscale
        path = folder / f"{start + i:06d}.png"
        Image.fromarray(array).save(path)
        paths.append(path)
    return paths


__all__ = ["image_grid", "save_grid", "save_images", "to_uint8"]


`anim_gen/runtime.py`: device, precision, multi-GPU helpers

In [ ]:
%%writefile anim_gen/runtime.py
"""Where and how training runs: seeding, device, precision, multi-GPU.

Multi-GPU / multi-node: launch the same command with torchrun, e.g.
    torchrun --nproc_per_node=4 -m anim_gen.train --preset cifar10-resnet
torchrun starts one process per GPU and sets RANK / WORLD_SIZE / LOCAL_RANK.

We do NOT wrap the networks in DistributedDataParallel. A GAN step runs the
discriminator several times per backward pass, and the gradient penalties
(WGAN-GP, R1) need a double backward; DDP supports neither well. Instead every
process computes its own gradients and `sync_gradients` averages them with one
all-reduce per optimizer step (the approach StyleGAN2-ADA uses). Weights start
equal (`broadcast_module`) and stay equal because every process then applies
the same averaged update.
"""

from __future__ import annotations

import os
import random
from collections.abc import Iterable, Iterator
from contextlib import AbstractContextManager, contextmanager
from dataclasses import dataclass

import torch
import torch.distributed as dist
from torch import nn

try:  # numpy is optional; we only need it to seed its RNG
    import numpy as np
except ImportError:  # pragma: no cover
    np = None


def seed_everything(seed: int) -> None:
    """Seed every random number generator we use."""
    random.seed(seed)
    if np is not None:
        np.random.seed(seed)
    torch.manual_seed(seed)  # also seeds all CUDA devices


def resolve_device(name: str = "auto") -> torch.device:
    """Turn "auto" | "cpu" | "cuda" | "mps" into a torch.device."""
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))  # under torchrun: cuda:LOCAL_RANK
    if name == "cuda" or (name == "auto" and torch.cuda.is_available()):
        return torch.device("cuda", local_rank)
    if name != "auto":
        return torch.device(name)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def resolve_precision(precision: str, device: torch.device) -> torch.dtype | None:
    """Autocast dtype, or None for plain fp32.

    "auto" picks bf16 only on GPUs that run it natively (Ampere and newer:
    A100, RTX 30xx/40xx, H100). Older GPUs such as the T4 / P100 on Kaggle and
    Colab stay in fp32. We do not offer fp16: GAN losses and gradient penalties
    overflow easily in fp16, and bf16 has the range of fp32.
    """
    if precision == "fp32":
        return None
    if precision == "bf16":
        return torch.bfloat16
    if precision == "auto":
        if device.type == "cuda" and torch.cuda.get_device_capability(device)[0] >= 8:
            return torch.bfloat16
        return None
    raise ValueError(f"unknown precision: {precision!r}")


def autocast(device: torch.device, dtype: torch.dtype | None) -> AbstractContextManager:
    """Mixed-precision context; does nothing when dtype is None."""
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=dtype is not None)


@dataclass(frozen=True)
class DistInfo:
    """Who am I in a multi-process run? Defaults describe a single process."""

    rank: int = 0  # global id of this process (0 .. world_size-1)
    world_size: int = 1  # total number of processes (= total GPUs)
    local_rank: int = 0  # id of this process on its own machine

    @property
    def enabled(self) -> bool:
        return self.world_size > 1

    @property
    def is_main(self) -> bool:
        """Only the main process logs, saves grids and writes checkpoints."""
        return self.rank == 0


def setup_distributed(device_name: str = "auto") -> DistInfo:
    """Join the process group if we were launched by torchrun; no-op otherwise."""
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    if world_size == 1:
        return DistInfo()
    local_rank = int(os.environ["LOCAL_RANK"])
    use_cuda = device_name in ("auto", "cuda") and torch.cuda.is_available()
    if use_cuda:
        torch.cuda.set_device(local_rank)
    # NCCL is the fast backend for GPUs; gloo works on CPU (used by the tests).
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


def get_dist_info() -> DistInfo:
    """Current process-group state (single process if none was started)."""
    if not (dist.is_available() and dist.is_initialized()):
        return DistInfo()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    return DistInfo(dist.get_rank(), dist.get_world_size(), local_rank)


@contextmanager
def local_main_first(info: DistInfo) -> Iterator[None]:
    """Let local rank 0 run the block first (downloads); the others then reuse the files."""
    if info.enabled and info.local_rank != 0:
        dist.barrier()
    yield
    if info.enabled and info.local_rank == 0:
        dist.barrier()


def broadcast_module(module: nn.Module) -> None:
    """Copy rank 0's parameters and buffers to every process."""
    if not get_dist_info().enabled:
        return
    with torch.no_grad():
        for tensor in list(module.parameters()) + list(module.buffers()):
            dist.broadcast(tensor.data, src=0)


def sync_gradients(parameters: Iterable[nn.Parameter]) -> None:
    """Average gradients over all processes with a single all-reduce."""
    info = get_dist_info()
    if not info.enabled:
        return
    grads = [p.grad for p in parameters if p.grad is not None]
    if not grads:
        return
    flat = torch.cat([g.reshape(-1) for g in grads])  # one big message is much faster
    dist.all_reduce(flat)
    flat /= info.world_size
    offset = 0
    for g in grads:
        g.copy_(flat[offset : offset + g.numel()].view_as(g))
        offset += g.numel()


def all_reduce_mean(values: dict[str, float], device: torch.device) -> dict[str, float]:
    """Average logged scalars over processes (no-op on one process)."""
    if not get_dist_info().enabled or not values:
        return values
    keys = sorted(values)
    tensor = torch.tensor([values[k] for k in keys], device=device, dtype=torch.float64)
    dist.all_reduce(tensor)
    tensor /= get_dist_info().world_size
    return dict(zip(keys, tensor.tolist(), strict=True))


__all__ = [
    "DistInfo",
    "all_reduce_mean",
    "autocast",
    "broadcast_module",
    "get_dist_info",
    "local_main_first",
    "resolve_device",
    "resolve_precision",
    "seed_everything",
    "setup_distributed",
    "sync_gradients",
]


`anim_gen/models/layers.py`: spectral norm, conditional BatchNorm, self-attention

In [ ]:
%%writefile anim_gen/models/layers.py
"""Building blocks shared by the GAN networks."""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn
from torch.nn.utils import parametrize
from torch.nn.utils.parametrizations import spectral_norm


def sn(module: nn.Module, enabled: bool = True) -> nn.Module:
    """Wrap a layer in spectral normalisation (largest singular value kept at 1).

    It bounds how fast the discriminator's output can change with its input
    (a Lipschitz constraint), the single most effective GAN stabiliser.
    """
    return spectral_norm(module) if enabled else module


def raw_weight(module: nn.Module) -> torch.Tensor:
    """The weight tensor to initialise in place (the original one under spectral norm)."""
    if parametrize.is_parametrized(module, "weight"):
        return module.parametrizations.weight.original
    return module.weight


def gan_widths(size: int, base: int, cap: int) -> dict[int, int]:
    """Channels per resolution: `base` at the top, doubling at each lower one, capped.

    gan_widths(32, 64, 512) -> {32: 64, 16: 128, 8: 256, 4: 512}
    """
    widths = {}
    r = size
    while r >= 4:
        widths[r] = min(cap, base * size // r)
        r //= 2
    return widths


class ConditionalBatchNorm2d(nn.Module):
    """BatchNorm whose scale and shift depend on the class label.

    With num_classes == 0 it is a plain BatchNorm2d. Each class gets its own
    (gamma, beta), initialised to (1, 0) so training starts like plain BN.
    """

    def __init__(self, channels: int, num_classes: int = 0) -> None:
        super().__init__()
        self.conditional = num_classes > 0
        self.bn = nn.BatchNorm2d(channels, affine=not self.conditional)
        if self.conditional:
            self.embed = nn.Embedding(num_classes, 2 * channels)
            nn.init.ones_(self.embed.weight[:, :channels])
            nn.init.zeros_(self.embed.weight[:, channels:])

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        out = self.bn(x)
        if not self.conditional:
            return out
        gamma, beta = self.embed(y).chunk(2, dim=1)
        return gamma[:, :, None, None] * out + beta[:, :, None, None]


class SelfAttention(nn.Module):
    """SAGAN self-attention: every pixel can look at every other pixel.

    Convolutions only see a small neighbourhood; attention lets the network
    keep far-apart parts consistent (two eyes of the same animal).
    The output is x + gamma * attention(x) with gamma starting at 0, so the
    block is the identity at first and the network learns how much to use it.
    """

    def __init__(self, channels: int, use_sn: bool = False) -> None:
        super().__init__()
        inner = max(1, channels // 8)
        self.query = sn(nn.Conv2d(channels, inner, 1, bias=False), use_sn)
        self.key = sn(nn.Conv2d(channels, inner, 1, bias=False), use_sn)
        self.value = sn(nn.Conv2d(channels, channels // 2, 1, bias=False), use_sn)
        self.out = sn(nn.Conv2d(channels // 2, channels, 1, bias=False), use_sn)
        self.gamma = nn.Parameter(torch.zeros(()))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        q = self.query(x).flatten(2).transpose(1, 2)  # [B, HW, c/8]
        k = self.key(x).flatten(2).transpose(1, 2)  # [B, HW, c/8]
        v = self.value(x).flatten(2).transpose(1, 2)  # [B, HW, c/2]
        attended = F.scaled_dot_product_attention(q[:, None], k[:, None], v[:, None])[:, 0]
        attended = attended.transpose(1, 2).reshape(b, c // 2, h, w)
        return x + self.gamma * self.out(attended)


__all__ = ["ConditionalBatchNorm2d", "SelfAttention", "gan_widths", "raw_weight", "sn"]


`anim_gen/models/dcgan.py`: DCGAN generator / discriminator

In [ ]:
%%writefile anim_gen/models/dcgan.py
"""DCGAN (Radford et al., 2015), for any power-of-two resolution.

Generator: noise z -> 4x4 feature map -> (transposed conv, BN, ReLU) x n -> tanh image.
Discriminator: image -> (strided conv, LeakyReLU) x n -> one realness logit.

The classic recipe, with two modern touches:
- the discriminator returns a raw logit (no sigmoid); losses.py applies the
  numerically stable form (softplus / BCE-with-logits) for each loss;
- optional class conditioning: label embedding concatenated to z (generator)
  and a projection term (discriminator, Miyato & Koyama 2018).
"""

from __future__ import annotations

import torch
from torch import nn

from anim_gen.config import ModelConfig
from anim_gen.models.layers import gan_widths, raw_weight, sn


def _dcgan_init(module: nn.Module) -> None:
    """N(0, 0.02) weights, as in the DCGAN paper."""
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear, nn.Embedding)):
        nn.init.normal_(raw_weight(module), 0.0, 0.02)
        if getattr(module, "bias", None) is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d) and module.affine:
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)


class DCGANGenerator(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        widths = gan_widths(config.image_size // 2, config.g_channels, config.max_channels)
        self.embed = nn.Embedding(config.num_classes, config.z_dim) if config.num_classes else None
        in_dim = config.z_dim * (2 if self.embed is not None else 1)

        layers: list[nn.Module] = [
            # [B, in_dim, 1, 1] -> [B, C4, 4, 4]
            nn.ConvTranspose2d(in_dim, widths[4], 4, 1, 0, bias=False),
            nn.BatchNorm2d(widths[4]),
            nn.ReLU(inplace=True),
        ]
        r = 4
        while r < config.image_size // 2:  # each block doubles the resolution
            layers += [
                nn.ConvTranspose2d(widths[r], widths[2 * r], 4, 2, 1, bias=False),
                nn.BatchNorm2d(widths[2 * r]),
                nn.ReLU(inplace=True),
            ]
            r *= 2
        layers += [nn.ConvTranspose2d(widths[r], config.channels, 4, 2, 1), nn.Tanh()]
        self.net = nn.Sequential(*layers)
        self.apply(_dcgan_init)

    def forward(self, z: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        """z [B, z_dim], y [B] class ids (or None) -> images [B, C, H, W] in [-1, 1]."""
        if self.embed is not None:
            z = torch.cat([z, self.embed(y)], dim=1)
        return self.net(z[:, :, None, None])


class DCGANDiscriminator(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        use_sn = config.spectral_norm
        widths = gan_widths(config.image_size // 2, config.d_channels, config.max_channels)
        r = config.image_size // 2
        layers: list[nn.Module] = [
            sn(nn.Conv2d(config.channels, widths[r], 4, 2, 1), use_sn),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        while r > 4:  # each block halves the resolution
            layers.append(sn(nn.Conv2d(widths[r], widths[r // 2], 4, 2, 1, bias=use_sn), use_sn))
            if not use_sn:  # DCGAN uses BN; with spectral norm we use no normalisation
                layers.append(nn.BatchNorm2d(widths[r // 2]))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            r //= 2
        self.features = nn.Sequential(*layers)
        feature_dim = widths[4] * 4 * 4
        self.head = sn(nn.Linear(feature_dim, 1), use_sn)
        self.embed = (
            sn(nn.Embedding(config.num_classes, feature_dim), use_sn)
            if config.num_classes
            else None
        )
        self.apply(_dcgan_init)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        """images [B, C, H, W], y [B] (or None) -> realness logits [B]."""
        h = self.features(x).flatten(1)
        out = self.head(h).squeeze(1)
        if self.embed is not None:  # projection: high when h matches the class embedding
            out = out + (self.embed(y) * h).sum(dim=1)
        return out


__all__ = ["DCGANDiscriminator", "DCGANGenerator"]


`anim_gen/models/resnet_gan.py`: ResNet GAN (SNGAN / BigGAN style)

In [ ]:
%%writefile anim_gen/models/resnet_gan.py
"""ResNet GAN in the style of SNGAN / SAGAN / BigGAN.

The strongest GAN recipe that still fits in one file:
- residual blocks in both networks (gradients flow through the skip path),
- spectral norm on every discriminator layer,
- class-conditional BatchNorm in the generator,
- projection discriminator for class conditioning,
- optional self-attention at one resolution (attn_resolution),
- trained with the hinge loss (losses.py), Adam(0, 0.9), and an EMA generator.

Generator   z -> Linear -> 4x4 -> GBlock(up) x n -> BN, ReLU, conv3 -> tanh
Discriminator   x -> DBlock(down) x n -> ReLU -> global sum pool -> Linear (+ projection)
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from torch import nn

from anim_gen.config import ModelConfig
from anim_gen.models.layers import (
    ConditionalBatchNorm2d,
    SelfAttention,
    gan_widths,
    raw_weight,
    sn,
)


def _orthogonal_init(module: nn.Module) -> None:
    """Orthogonal init (BigGAN): keeps signal scale stable through deep stacks."""
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.orthogonal_(raw_weight(module))
        if module.bias is not None:
            nn.init.zeros_(module.bias)


class GBlock(nn.Module):
    """BN -> ReLU -> upsample -> conv3 -> BN -> ReLU -> conv3, plus an upsampled skip."""

    def __init__(self, in_ch: int, out_ch: int, num_classes: int) -> None:
        super().__init__()
        self.bn1 = ConditionalBatchNorm2d(in_ch, num_classes)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.bn2 = ConditionalBatchNorm2d(out_ch, num_classes)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None) -> torch.Tensor:
        h = F.relu(self.bn1(x, y))
        h = F.interpolate(h, scale_factor=2, mode="nearest")
        h = self.conv1(h)
        h = self.conv2(F.relu(self.bn2(h, y)))
        return h + self.skip(F.interpolate(x, scale_factor=2, mode="nearest"))


class DBlock(nn.Module):
    """(ReLU) -> conv3 -> ReLU -> conv3 -> avgpool, plus a pooled 1x1 skip.

    The first block skips the leading ReLU: its input is the raw image.
    """

    def __init__(self, in_ch: int, out_ch: int, use_sn: bool, first: bool = False) -> None:
        super().__init__()
        self.first = first
        self.conv1 = sn(nn.Conv2d(in_ch, out_ch, 3, padding=1), use_sn)
        self.conv2 = sn(nn.Conv2d(out_ch, out_ch, 3, padding=1), use_sn)
        self.skip = sn(nn.Conv2d(in_ch, out_ch, 1), use_sn)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = x if self.first else F.relu(x)
        h = self.conv2(F.relu(self.conv1(h)))
        h = F.avg_pool2d(h, 2)
        if self.first:  # pool first: the 1x1 conv then runs on 4x fewer pixels
            return h + self.skip(F.avg_pool2d(x, 2))
        return h + F.avg_pool2d(self.skip(x), 2)


class ResNetGenerator(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        widths = gan_widths(config.image_size, config.g_channels, config.max_channels)
        self.proj = nn.Linear(config.z_dim, widths[4] * 4 * 4)
        self.blocks = nn.ModuleList()
        attn = {}
        r = 4
        while r < config.image_size:
            self.blocks.append(GBlock(widths[r], widths[2 * r], config.num_classes))
            r *= 2
            if r == config.attn_resolution:
                attn[str(len(self.blocks) - 1)] = SelfAttention(widths[r])
        self.attn = nn.ModuleDict(attn)  # keyed by the index of the block it follows
        self.out_bn = nn.BatchNorm2d(widths[config.image_size])
        self.out_conv = nn.Conv2d(widths[config.image_size], config.channels, 3, padding=1)
        self.apply(_orthogonal_init)

    def forward(self, z: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        """z [B, z_dim], y [B] class ids (or None) -> images [B, C, H, W] in [-1, 1]."""
        h = self.proj(z).view(z.shape[0], -1, 4, 4)
        for i, block in enumerate(self.blocks):
            h = block(h, y)
            if str(i) in self.attn:
                h = self.attn[str(i)](h)
        return torch.tanh(self.out_conv(F.relu(self.out_bn(h))))


class ResNetDiscriminator(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        use_sn = config.spectral_norm
        widths = gan_widths(config.image_size // 2, config.d_channels, config.max_channels)
        blocks: list[nn.Module] = []
        in_ch, r = config.channels, config.image_size
        while r > 4:
            r //= 2
            blocks.append(DBlock(in_ch, widths[r], use_sn, first=not blocks))
            in_ch = widths[r]
            if r == config.attn_resolution:
                blocks.append(SelfAttention(in_ch, use_sn=use_sn))
        self.blocks = nn.Sequential(*blocks)
        self.head = sn(nn.Linear(in_ch, 1), use_sn)
        self.embed = (
            sn(nn.Embedding(config.num_classes, in_ch), use_sn) if config.num_classes else None
        )
        self.apply(_orthogonal_init)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        """images [B, C, H, W], y [B] (or None) -> realness logits [B]."""
        h = F.relu(self.blocks(x)).sum(dim=(2, 3))  # global sum pooling -> [B, C]
        out = self.head(h).squeeze(1)
        if self.embed is not None:  # projection discriminator
            out = out + (self.embed(y) * h).sum(dim=1)
        return out


__all__ = ["ResNetDiscriminator", "ResNetGenerator"]


`anim_gen/models/unet.py`: diffusion U-Net

In [ ]:
%%writefile anim_gen/models/unet.py
"""U-Net denoiser for diffusion (DDPM, Ho et al. 2020; improvements from
Nichol & Dhariwal 2021 and Dhariwal & Nichol 2021).

Input: a noisy image x_t, the timestep t and (optionally) a class label.
Output: a tensor shaped like the image: the predicted noise (or velocity).

    x_t -> conv3 -> [ResBlock x n (+attn) -> down] per level -> middle
        -> [ResBlock x (n+1) with skip concat (+attn) -> up] per level -> conv3

Every ResBlock receives the time (+ class) embedding through a FiLM-style
scale and shift, so the same network can denoise at every noise level.
The label index `num_classes` is the "no label" token used for
classifier-free guidance.
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn

from anim_gen.config import ModelConfig


def timestep_embedding(t: torch.Tensor, dim: int, max_period: float = 10_000.0) -> torch.Tensor:
    """Sinusoidal embedding of (possibly fractional) timesteps, [B] -> [B, dim]."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=1)


def group_norm(channels: int) -> nn.GroupNorm:
    """GroupNorm with up to 32 groups (works for small test widths too)."""
    groups = math.gcd(32, channels)
    return nn.GroupNorm(groups, channels)


def zero_module(module: nn.Module) -> nn.Module:
    """Zero-init a layer so the residual branch starts as the identity."""
    for p in module.parameters():
        nn.init.zeros_(p)
    return module


class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, emb_dim: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = group_norm(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.emb = nn.Linear(emb_dim, 2 * out_ch)  # -> (scale, shift)
        self.norm2 = group_norm(out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = zero_module(nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.emb(F.silu(emb))[:, :, None, None].chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.conv2(self.dropout(F.silu(h)))
        return h + self.skip(x)


class AttentionBlock(nn.Module):
    """Multi-head self-attention over all pixels, with a residual connection."""

    def __init__(self, channels: int, head_dim: int = 64) -> None:
        super().__init__()
        self.heads = max(1, channels // head_dim)
        self.norm = group_norm(channels)
        self.qkv = nn.Conv2d(channels, 3 * channels, 1)
        self.out = zero_module(nn.Conv2d(channels, channels, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        q, k, v = self.qkv(self.norm(x)).reshape(b, 3, self.heads, c // self.heads, h * w).unbind(1)
        # [B, heads, HW, head_dim] for scaled_dot_product_attention
        out = F.scaled_dot_product_attention(
            q.transpose(-1, -2), k.transpose(-1, -2), v.transpose(-1, -2)
        )
        out = out.transpose(-1, -2).reshape(b, c, h, w)
        return x + self.out(out)


class Downsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))


class UNet(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        ch = config.unet_channels
        emb_dim = 4 * ch
        self.time_mlp = nn.Sequential(
            nn.Linear(ch, emb_dim), nn.SiLU(), nn.Linear(emb_dim, emb_dim)
        )
        # One extra row: index num_classes = "no label" (classifier-free guidance).
        self.class_embed = (
            nn.Embedding(config.num_classes + 1, emb_dim) if config.num_classes else None
        )
        self.conv_in = nn.Conv2d(config.channels, ch, 3, padding=1)

        # --- encoder: remember the width of every skip connection ---
        self.down = nn.ModuleList()
        skip_widths = [ch]
        width, res = ch, config.image_size
        for level, mult in enumerate(config.channel_mult):
            for _ in range(config.num_res_blocks):
                out = ch * mult
                layers = [ResBlock(width, out, emb_dim, config.dropout)]
                if res in config.attn_resolutions:
                    layers.append(AttentionBlock(out))
                self.down.append(nn.ModuleList(layers))
                width = out
                skip_widths.append(width)
            if level != len(config.channel_mult) - 1:
                self.down.append(nn.ModuleList([Downsample(width)]))
                skip_widths.append(width)
                res //= 2

        self.mid = nn.ModuleList(
            [
                ResBlock(width, width, emb_dim, config.dropout),
                AttentionBlock(width),
                ResBlock(width, width, emb_dim, config.dropout),
            ]
        )

        # --- decoder: one more block per level, each eats one skip ---
        self.up = nn.ModuleList()
        for level, mult in reversed(list(enumerate(config.channel_mult))):
            for i in range(config.num_res_blocks + 1):
                out = ch * mult
                layers = [ResBlock(width + skip_widths.pop(), out, emb_dim, config.dropout)]
                if res in config.attn_resolutions:
                    layers.append(AttentionBlock(out))
                if level != 0 and i == config.num_res_blocks:
                    layers.append(Upsample(out))
                    res *= 2
                self.up.append(nn.ModuleList(layers))
                width = out
        assert not skip_widths

        self.out = nn.Sequential(
            group_norm(width),
            nn.SiLU(),
            zero_module(nn.Conv2d(width, config.channels, 3, padding=1)),
        )

    def forward(
        self, x: torch.Tensor, t: torch.Tensor, y: torch.Tensor | None = None
    ) -> torch.Tensor:
        """x [B, C, H, W], t [B] timesteps, y [B] labels (num_classes = none) -> [B, C, H, W]."""
        emb = self.time_mlp(timestep_embedding(t, self.config.unet_channels))
        if self.class_embed is not None:
            if y is None:  # unconditional request on a conditional model
                y = torch.full((x.shape[0],), self.config.num_classes, device=x.device)
            emb = emb + self.class_embed(y)

        h = self.conv_in(x)
        skips = [h]
        for layers in self.down:
            for layer in layers:
                h = layer(h, emb) if isinstance(layer, ResBlock) else layer(h)
            skips.append(h)
        for layer in self.mid:
            h = layer(h, emb) if isinstance(layer, ResBlock) else layer(h)
        for layers in self.up:
            h = torch.cat([h, skips.pop()], dim=1)
            for layer in layers:
                h = layer(h, emb) if isinstance(layer, ResBlock) else layer(h)
        return self.out(h)


__all__ = ["UNet", "timestep_embedding"]


`anim_gen/models/__init__.py`: architecture registry

In [ ]:
%%writefile anim_gen/models/__init__.py
"""Network architectures and a small registry to build them from a ModelConfig.

arch     family      networks
dcgan    GAN         DCGANGenerator / DCGANDiscriminator
resnet   GAN         ResNetGenerator / ResNetDiscriminator (SNGAN / BigGAN style)
unet     diffusion   UNet denoiser (DDPM / improved DDPM)
"""

from __future__ import annotations

from torch import nn

from anim_gen.config import ModelConfig
from anim_gen.models.dcgan import DCGANDiscriminator, DCGANGenerator
from anim_gen.models.resnet_gan import ResNetDiscriminator, ResNetGenerator
from anim_gen.models.unet import UNet

GENERATORS = {"dcgan": DCGANGenerator, "resnet": ResNetGenerator}
DISCRIMINATORS = {"dcgan": DCGANDiscriminator, "resnet": ResNetDiscriminator}


def build_generator(config: ModelConfig) -> nn.Module:
    return GENERATORS[config.arch](config)


def build_discriminator(config: ModelConfig) -> nn.Module:
    return DISCRIMINATORS[config.arch](config)


def build_unet(config: ModelConfig) -> nn.Module:
    return UNet(config)


def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


__all__ = [
    "DCGANDiscriminator",
    "DCGANGenerator",
    "ResNetDiscriminator",
    "ResNetGenerator",
    "UNet",
    "build_discriminator",
    "build_generator",
    "build_unet",
    "count_parameters",
]


`anim_gen/losses.py`: GAN losses, gradient penalty, R1

In [ ]:
%%writefile anim_gen/losses.py
"""GAN losses and regularisers. Discriminators output raw logits (no sigmoid).

    loss            discriminator                          generator
    nonsaturating   softplus(-D(x)) + softplus(D(G(z)))    softplus(-D(G(z)))
    hinge           relu(1 - D(x)) + relu(1 + D(G(z)))     -D(G(z))
    wgan-gp         D(G(z)) - D(x)  (+ gradient penalty)   -D(G(z))
    lsgan           (D(x) - 1)^2 + D(G(z))^2  (/2)         (D(G(z)) - 1)^2 / 2

softplus(-l) is exactly BCE(sigmoid(l), 1), computed without overflow.
"Non-saturating": the generator maximises log D(G(z)) instead of minimising
log(1 - D(G(z))), which has vanishing gradients when D wins early on.
"""

from __future__ import annotations

from collections.abc import Callable

import torch
import torch.nn.functional as F


def d_loss(kind: str, real_logits: torch.Tensor, fake_logits: torch.Tensor) -> torch.Tensor:
    """Discriminator loss (to minimise), averaged over the batch."""
    if kind == "nonsaturating":
        return F.softplus(-real_logits).mean() + F.softplus(fake_logits).mean()
    if kind == "hinge":
        return F.relu(1 - real_logits).mean() + F.relu(1 + fake_logits).mean()
    if kind == "wgan-gp":
        return fake_logits.mean() - real_logits.mean()
    if kind == "lsgan":
        return 0.5 * ((real_logits - 1) ** 2).mean() + 0.5 * (fake_logits**2).mean()
    raise ValueError(f"unknown GAN loss: {kind!r}")


def g_loss(kind: str, fake_logits: torch.Tensor) -> torch.Tensor:
    """Generator loss (to minimise), averaged over the batch."""
    if kind == "nonsaturating":
        return F.softplus(-fake_logits).mean()
    if kind in ("hinge", "wgan-gp"):
        return -fake_logits.mean()
    if kind == "lsgan":
        return 0.5 * ((fake_logits - 1) ** 2).mean()
    raise ValueError(f"unknown GAN loss: {kind!r}")


def gradient_penalty(
    critic: Callable[[torch.Tensor], torch.Tensor], real: torch.Tensor, fake: torch.Tensor
) -> torch.Tensor:
    """WGAN-GP (Gulrajani et al. 2017): E[(||grad D(x_hat)|| - 1)^2].

    x_hat is a random point on the line between a real and a fake image; the
    penalty keeps the critic 1-Lipschitz there.
    """
    alpha = torch.rand(real.shape[0], 1, 1, 1, device=real.device)
    mixed = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    (grad,) = torch.autograd.grad(critic(mixed).sum(), mixed, create_graph=True)
    return ((grad.flatten(1).norm(dim=1) - 1) ** 2).mean()


def r1_penalty(real_logits: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
    """R1 (Mescheder et al. 2018): E[||grad D(x)||^2] on real images only.

    `real` must have requires_grad=True before D was called on it.
    Use as  loss + gamma / 2 * r1_penalty(...)  (StyleGAN convention).
    """
    (grad,) = torch.autograd.grad(real_logits.sum(), real, create_graph=True)
    return grad.flatten(1).pow(2).sum(dim=1).mean()


__all__ = ["d_loss", "g_loss", "gradient_penalty", "r1_penalty"]


`anim_gen/diffaugment.py`: DiffAugment

In [ ]:
%%writefile anim_gen/diffaugment.py
"""DiffAugment (Zhao et al. 2020): differentiable augmentation for data-efficient GANs.

With a few thousand images (AFHQ has ~5k per class) the discriminator
memorises the training set and training collapses. Augmenting *both* real and
fake images, with operations gradients can flow through, stops that without
the generator learning to produce augmented images.

    policy "color"       random brightness, saturation, contrast
    policy "translation" random shift by up to 1/8 of the image, zero padded
    policy "cutout"      zero a random square of half the image size
"""

from __future__ import annotations

import torch
import torch.nn.functional as F


def rand_brightness(x: torch.Tensor) -> torch.Tensor:
    return x + (torch.rand(x.shape[0], 1, 1, 1, device=x.device) - 0.5)


def rand_saturation(x: torch.Tensor) -> torch.Tensor:
    mean = x.mean(dim=1, keepdim=True)
    return (x - mean) * (torch.rand(x.shape[0], 1, 1, 1, device=x.device) * 2) + mean


def rand_contrast(x: torch.Tensor) -> torch.Tensor:
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    return (x - mean) * (torch.rand(x.shape[0], 1, 1, 1, device=x.device) + 0.5) + mean


def rand_translation(x: torch.Tensor, ratio: float = 0.125) -> torch.Tensor:
    b, _, h, w = x.shape
    sh, sw = int(h * ratio + 0.5), int(w * ratio + 0.5)
    shift_y = torch.randint(-sh, sh + 1, (b, 1, 1), device=x.device)
    shift_x = torch.randint(-sw, sw + 1, (b, 1, 1), device=x.device)
    grid_y, grid_x = torch.meshgrid(
        torch.arange(h, device=x.device), torch.arange(w, device=x.device), indexing="ij"
    )
    # +1 because of the 1-pixel zero padding below; out-of-range reads hit the padding.
    y = torch.clamp(grid_y[None] + shift_y + 1, 0, h + 1)
    xs = torch.clamp(grid_x[None] + shift_x + 1, 0, w + 1)
    padded = F.pad(x, [1, 1, 1, 1])
    batch = torch.arange(b, device=x.device)[:, None, None]
    return padded.permute(0, 2, 3, 1)[batch, y, xs].permute(0, 3, 1, 2).contiguous()


def rand_cutout(x: torch.Tensor, ratio: float = 0.5) -> torch.Tensor:
    b, _, h, w = x.shape
    ch, cw = int(h * ratio + 0.5), int(w * ratio + 0.5)
    # top-left corner of the cut square; it may stick out of the image
    top = torch.randint(0, h + 1 - ch % 2, (b, 1, 1), device=x.device) - ch // 2
    left = torch.randint(0, w + 1 - cw % 2, (b, 1, 1), device=x.device) - cw // 2
    grid_y, grid_x = torch.meshgrid(
        torch.arange(h, device=x.device), torch.arange(w, device=x.device), indexing="ij"
    )
    cut = (grid_y >= top) & (grid_y < top + ch) & (grid_x >= left) & (grid_x < left + cw)
    return x * (~cut)[:, None].to(x.dtype)


POLICIES = {
    "color": (rand_brightness, rand_saturation, rand_contrast),
    "translation": (rand_translation,),
    "cutout": (rand_cutout,),
}


def parse_policy(policy: str) -> list[str]:
    """ "color,translation" -> ["color", "translation"]; validates the names."""
    names = [p.strip() for p in policy.split(",") if p.strip()]
    unknown = set(names) - set(POLICIES)
    if unknown:
        raise ValueError(f"unknown DiffAugment policy {sorted(unknown)}; use {sorted(POLICIES)}")
    return names


def diff_augment(x: torch.Tensor, policy: str) -> torch.Tensor:
    """Apply the policy to a batch in [-1, 1]. Empty policy = identity."""
    for name in parse_policy(policy):
        for fn in POLICIES[name]:
            x = fn(x)
    return x


__all__ = ["POLICIES", "diff_augment", "parse_policy"]


`anim_gen/ema.py`: EMA of the weights

In [ ]:
%%writefile anim_gen/ema.py
"""Exponential moving average (EMA) of model weights.

    ema = decay * ema + (1 - decay) * weights      after every optimizer step

The averaged model is smoother than the raw one; for both GANs and diffusion
it gives clearly better samples and FID, so sampling always uses it.
"""

from __future__ import annotations

import copy

import torch
from torch import nn


class EMA(nn.Module):
    """Holds a frozen copy of `model` and moves it towards the live weights."""

    def __init__(self, model: nn.Module, decay: float = 0.999, start: int = 0) -> None:
        super().__init__()
        self.decay = decay
        self.start = start  # before this step the copy simply follows the model
        self.model = copy.deepcopy(model).eval().requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module, step: int) -> None:
        decay = self.decay if step >= self.start else 0.0
        for ema_p, p in zip(self.model.parameters(), model.parameters(), strict=True):
            ema_p.lerp_(p.detach(), 1.0 - decay)
        # Buffers (BatchNorm running statistics, spectral-norm vectors) are copied.
        for ema_b, b in zip(self.model.buffers(), model.buffers(), strict=True):
            ema_b.copy_(b)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)


__all__ = ["EMA"]


`anim_gen/diffusion.py`: noise schedule, DDIM / DDPM sampler, guidance

In [ ]:
%%writefile anim_gen/diffusion.py
"""Gaussian diffusion: how images are noised (training) and denoised (sampling).

Forward process, fixed:  x_t = sqrt(a_t) * x_0 + sqrt(1 - a_t) * noise,
where a_t ("alpha bar") falls from ~1 (t = 0, clean) to ~0 (t = T-1, pure noise).

Training: pick a random t, noise the image, ask the U-Net to predict
- "eps": the noise that was added (DDPM), or
- "v":   sqrt(a_t) * noise - sqrt(1 - a_t) * x_0 (Salimans & Ho 2022; better at high res).
The loss is a plain mean squared error.

Sampling: start from pure noise and step backwards with DDIM (Song et al. 2021).
eta = 0 is deterministic and works with few steps (50-250); eta = 1 with all
T steps is the original DDPM sampler. Classifier-free guidance (Ho & Salimans
2022) mixes the conditional and unconditional predictions:
    pred = uncond + guidance_scale * (cond - uncond)
"""

from __future__ import annotations

import math

import torch
import torch.nn.functional as F
from torch import nn


def make_alpha_bar(timesteps: int, schedule: str = "cosine") -> torch.Tensor:
    """a_t for t = 0 .. T-1 (float64)."""
    if schedule == "linear":
        scale = 1000 / timesteps  # keeps the DDPM endpoints for any T
        betas = torch.linspace(scale * 1e-4, scale * 0.02, timesteps, dtype=torch.float64)
    elif schedule == "cosine":  # Nichol & Dhariwal: no wasted steps near pure noise
        s = 0.008
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        betas = (1 - f[1:] / f[:-1]).clamp(max=0.999)
    else:
        raise ValueError(f"unknown noise schedule: {schedule!r}")
    return torch.cumprod(1 - betas, dim=0)


def _expand(values: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """Pick values[t] per sample and shape it [B, 1, 1, 1] to broadcast over images."""
    return values[t].float()[:, None, None, None]


class GaussianDiffusion(nn.Module):
    """Holds the noise schedule (as buffers, so .to(device) moves it)."""

    def __init__(
        self,
        timesteps: int = 1000,
        schedule: str = "cosine",
        prediction: str = "eps",
        null_label: int | None = None,
    ) -> None:
        super().__init__()
        self.timesteps = timesteps
        self.prediction = prediction
        self.null_label = null_label  # label id meaning "no class" (None = unconditional)
        self.register_buffer("alpha_bar", make_alpha_bar(timesteps, schedule), persistent=False)

    # --- training ---------------------------------------------------------

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        """Noise clean images x0 to level t."""
        a = _expand(self.alpha_bar, t)
        return a.sqrt() * x0 + (1 - a).sqrt() * noise

    def target(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        """What the network should output for (x0, t, noise)."""
        if self.prediction == "eps":
            return noise
        a = _expand(self.alpha_bar, t)
        return a.sqrt() * noise - (1 - a).sqrt() * x0

    def training_loss(
        self, model: nn.Module, x0: torch.Tensor, y: torch.Tensor | None = None
    ) -> torch.Tensor:
        """MSE between the network output and its target at a random noise level."""
        t = torch.randint(0, self.timesteps, (x0.shape[0],), device=x0.device)
        noise = torch.randn_like(x0)
        prediction = model(self.q_sample(x0, t, noise), t, y)
        return F.mse_loss(prediction.float(), self.target(x0, t, noise))

    # --- sampling ---------------------------------------------------------

    def predict_x0_eps(
        self,
        model: nn.Module,
        x: torch.Tensor,
        t: torch.Tensor,
        y: torch.Tensor | None,
        guidance_scale: float = 1.0,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Network output -> (clean image estimate clipped to [-1, 1], matching noise)."""
        if y is not None and guidance_scale != 1.0 and self.null_label is not None:
            # One forward on [cond; uncond] instead of two.
            null = torch.full_like(y, self.null_label)
            out = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([y, null])).float()
            cond, uncond = out.chunk(2)
            out = uncond + guidance_scale * (cond - uncond)
        else:
            out = model(x, t, y).float()
        a = _expand(self.alpha_bar, t)
        if self.prediction == "eps":
            x0 = (x - (1 - a).sqrt() * out) / a.sqrt()
        else:  # v
            x0 = a.sqrt() * x - (1 - a).sqrt() * out
        x0 = x0.clamp(-1, 1)
        eps = (x - a.sqrt() * x0) / (1 - a).sqrt()  # consistent with the clipped x0
        return x0, eps

    def sample_timesteps(self, steps: int) -> list[int]:
        """`steps` timesteps from T-1 down to 0, evenly spaced."""
        steps = max(1, min(steps, self.timesteps))
        ts = torch.linspace(self.timesteps - 1, 0, steps).round().long().tolist()
        return list(dict.fromkeys(ts))  # drop duplicates, keep order

    @torch.no_grad()
    def sample(
        self,
        model: nn.Module,
        noise: torch.Tensor,
        y: torch.Tensor | None = None,
        steps: int = 100,
        eta: float = 0.0,
        guidance_scale: float = 1.0,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        """Denoise `noise` [B, C, H, W] into images in [-1, 1] with DDIM."""
        x = noise
        ts = self.sample_timesteps(steps)
        for i, t_now in enumerate(ts):
            t = torch.full((x.shape[0],), t_now, device=x.device, dtype=torch.long)
            x0, eps = self.predict_x0_eps(model, x, t, y, guidance_scale)
            a = self.alpha_bar[t_now].float()
            a_prev = self.alpha_bar[ts[i + 1]].float() if i + 1 < len(ts) else x.new_tensor(1.0)
            sigma = eta * ((1 - a_prev) / (1 - a) * (1 - a / a_prev)).clamp(min=0).sqrt()
            x = a_prev.sqrt() * x0 + (1 - a_prev - sigma**2).clamp(min=0).sqrt() * eps
            if eta > 0 and i + 1 < len(ts):
                z = torch.randn(x.shape, device=x.device, generator=generator)
                x = x + sigma * z
        return x.clamp(-1, 1)


__all__ = ["GaussianDiffusion", "make_alpha_bar"]


`anim_gen/tasks.py`: GANTask / DiffusionTask

In [ ]:
%%writefile anim_gen/tasks.py
"""The two ways of learning to generate images, behind one interface.

    GANTask        generator vs discriminator (arch "dcgan" or "resnet")
    DiffusionTask  U-Net denoiser + Gaussian diffusion (arch "unet")

Both expose the same methods, so train.py, sample.py and fid.py never need
to know which family they are driving:

    task.train_step(batches, step) -> {"metric": tensor}   one optimizer step
    task.noise(n, generator)       -> the random input of the sampler
    task.generate(noise, labels)   -> images in [-1, 1] (always from the EMA weights)
    task.state_dict() / task.load_state_dict(state)
"""

from __future__ import annotations

from collections.abc import Iterator

import torch
from torch import nn
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import LambdaLR

from anim_gen.config import ModelConfig, TrainConfig
from anim_gen.diffaugment import diff_augment
from anim_gen.diffusion import GaussianDiffusion
from anim_gen.ema import EMA
from anim_gen.losses import d_loss, g_loss, gradient_penalty, r1_penalty
from anim_gen.models import build_discriminator, build_generator, build_unet
from anim_gen.runtime import autocast, broadcast_module, sync_gradients

Batches = Iterator[tuple[torch.Tensor, torch.Tensor]]


class Task:
    """Shared plumbing. Subclasses fill `networks` and `optimizers`."""

    def __init__(
        self,
        model_config: ModelConfig,
        train_config: TrainConfig | None,
        device: torch.device,
        amp_dtype: torch.dtype | None = None,
    ) -> None:
        self.model_config = model_config
        self.train_config = train_config  # None = inference only (no optimizers)
        self.device = device
        self.amp_dtype = amp_dtype
        self.networks: dict[str, nn.Module] = {}
        self.optimizers: dict[str, torch.optim.Optimizer] = {}
        self.schedulers: dict[str, LambdaLR] = {}

    @property
    def conditional(self) -> bool:
        return self.model_config.num_classes > 0

    def _next(self, batches: Batches) -> tuple[torch.Tensor, torch.Tensor | None]:
        images, labels = next(batches)
        images = images.to(self.device, non_blocking=True)
        labels = labels.to(self.device, non_blocking=True) if self.conditional else None
        return images, labels

    def random_labels(self, n: int, generator: torch.Generator | None = None):
        """Uniform class ids (or None for an unconditional model)."""
        if not self.conditional:
            return None
        return torch.randint(0, self.model_config.num_classes, (n,), generator=generator).to(
            self.device
        )

    def state_dict(self) -> dict:
        return {
            "networks": {k: v.state_dict() for k, v in self.networks.items()},
            "optimizers": {k: v.state_dict() for k, v in self.optimizers.items()},
            "schedulers": {k: v.state_dict() for k, v in self.schedulers.items()},
        }

    def load_state_dict(self, state: dict) -> None:
        """Load what this task has; inference tasks simply skip optimizer states."""
        for name, net in self.networks.items():
            net.load_state_dict(state["networks"][name])
        for name, opt in self.optimizers.items():
            opt.load_state_dict(state["optimizers"][name])
        for name, sched in self.schedulers.items():
            sched.load_state_dict(state["schedulers"][name])


class GANTask(Task):
    """Alternate `d_steps` discriminator updates and one generator update."""

    family = "gan"

    def __init__(self, model_config, train_config, device, amp_dtype=None) -> None:
        super().__init__(model_config, train_config, device, amp_dtype)
        self.G = build_generator(model_config).to(device)
        self.networks["G"] = self.G
        cfg = train_config
        if cfg is None:  # inference: the checkpoint's EMA weights are loaded into G_ema
            self.G_ema = EMA(self.G, 0.0)
            self.networks["G_ema"] = self.G_ema
            return
        self.D = build_discriminator(model_config).to(device)
        broadcast_module(self.G)  # every process starts from rank 0's weights
        broadcast_module(self.D)
        self.G_ema = EMA(self.G, cfg.ema_decay, cfg.ema_start)
        self.networks.update(D=self.D, G_ema=self.G_ema)
        betas = (cfg.beta1, cfg.beta2)
        self.optimizers["G"] = Adam(self.G.parameters(), lr=cfg.lr_g, betas=betas)
        self.optimizers["D"] = Adam(self.D.parameters(), lr=cfg.lr_d, betas=betas)

    def noise(self, n: int, generator: torch.Generator | None = None) -> torch.Tensor:
        return torch.randn(n, self.model_config.z_dim, generator=generator).to(self.device)

    def _discriminator_step(self, batches: Batches) -> dict[str, torch.Tensor]:
        cfg = self.train_config
        real, y = self._next(batches)
        with torch.no_grad(), autocast(self.device, self.amp_dtype):
            fake = self.G(self.noise(real.shape[0]), y)
        real_in = diff_augment(real, cfg.diffaug)
        fake_in = diff_augment(fake.float(), cfg.diffaug)
        if cfg.r1_gamma > 0:
            real_in = real_in.detach().requires_grad_(True)
        with autocast(self.device, self.amp_dtype):
            real_logits = self.D(real_in, y).float()
            fake_logits = self.D(fake_in, y).float()
        loss = d_loss(cfg.loss, real_logits, fake_logits)
        metrics = {
            "d_loss": loss.detach(),
            "d_real": real_logits.detach().mean(),  # mean logit on real images
            "d_fake": fake_logits.detach().mean(),  # ... and on generated ones
        }
        if cfg.loss == "wgan-gp":  # in fp32: second-order gradients are precision-hungry
            gp = gradient_penalty(lambda x: self.D(x, y), real_in.detach(), fake_in)
            loss = loss + cfg.gp_weight * gp
            metrics["gp"] = gp.detach()
        if cfg.r1_gamma > 0:
            r1 = r1_penalty(real_logits, real_in)
            loss = loss + cfg.r1_gamma / 2 * r1
            metrics["r1"] = r1.detach()

        opt = self.optimizers["D"]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        sync_gradients(self.D.parameters())
        opt.step()
        return metrics

    def train_step(self, batches: Batches, step: int) -> dict[str, torch.Tensor]:
        cfg = self.train_config
        for _ in range(cfg.d_steps):
            metrics = self._discriminator_step(batches)

        # Generator: fool the (now frozen) discriminator on fresh noise.
        n = cfg.batch_size
        y = self.random_labels(n)
        self.D.requires_grad_(False)  # no gradients wasted on D's weights
        with autocast(self.device, self.amp_dtype):
            fake = self.G(self.noise(n), y)
            fake_logits = self.D(diff_augment(fake, cfg.diffaug), y).float()
        loss = g_loss(cfg.loss, fake_logits)
        opt = self.optimizers["G"]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        sync_gradients(self.G.parameters())
        opt.step()
        self.D.requires_grad_(True)
        self.G_ema.update(self.G, step)
        metrics["g_loss"] = loss.detach()
        return metrics

    @torch.no_grad()
    def generate(
        self, noise: torch.Tensor, labels: torch.Tensor | None = None, **_
    ) -> torch.Tensor:
        with autocast(self.device, self.amp_dtype):
            return self.G_ema(noise, labels).float()


class DiffusionTask(Task):
    """Denoising diffusion: one MSE regression step per batch."""

    family = "diffusion"

    def __init__(self, model_config, train_config, device, amp_dtype=None) -> None:
        super().__init__(model_config, train_config, device, amp_dtype)
        self.unet = build_unet(model_config).to(device)
        self.diffusion = GaussianDiffusion(
            model_config.timesteps,
            model_config.noise_schedule,
            model_config.prediction,
            null_label=model_config.num_classes if self.conditional else None,
        ).to(device)
        self.networks["unet"] = self.unet
        cfg = train_config
        if cfg is None:
            self.ema = EMA(self.unet, 0.0)
            self.networks["ema"] = self.ema
            return
        broadcast_module(self.unet)
        self.ema = EMA(self.unet, cfg.ema_decay, cfg.ema_start)
        self.networks["ema"] = self.ema
        self.optimizers["unet"] = AdamW(
            self.unet.parameters(),
            lr=cfg.lr,
            betas=(cfg.beta1, cfg.beta2),
            weight_decay=cfg.weight_decay,
        )
        warmup = max(1, cfg.warmup_steps)  # linear warmup, then constant (as in DDPM)
        self.schedulers["unet"] = LambdaLR(
            self.optimizers["unet"], lambda s: min(1.0, (s + 1) / warmup)
        )

    def noise(self, n: int, generator: torch.Generator | None = None) -> torch.Tensor:
        c, s = self.model_config.channels, self.model_config.image_size
        return torch.randn(n, c, s, s, generator=generator).to(self.device)

    def train_step(self, batches: Batches, step: int) -> dict[str, torch.Tensor]:
        cfg = self.train_config
        x0, y = self._next(batches)
        if y is not None and self.model_config.cfg_dropout > 0:
            # Sometimes hide the label, so the same network also learns p(x):
            # that is what classifier-free guidance needs at sampling time.
            drop = torch.rand(y.shape[0], device=y.device) < self.model_config.cfg_dropout
            y = torch.where(drop, torch.full_like(y, self.model_config.num_classes), y)
        with autocast(self.device, self.amp_dtype):
            loss = self.diffusion.training_loss(self.unet, x0, y)
        opt = self.optimizers["unet"]
        opt.zero_grad(set_to_none=True)
        loss.backward()
        sync_gradients(self.unet.parameters())
        max_norm = cfg.grad_clip if cfg.grad_clip > 0 else float("inf")
        grad_norm = nn.utils.clip_grad_norm_(self.unet.parameters(), max_norm)
        opt.step()
        self.schedulers["unet"].step()
        self.ema.update(self.unet, step)
        return {"loss": loss.detach(), "grad_norm": grad_norm.detach()}

    @torch.no_grad()
    def generate(
        self,
        noise: torch.Tensor,
        labels: torch.Tensor | None = None,
        steps: int = 100,
        eta: float = 0.0,
        guidance_scale: float = 1.0,
        generator: torch.Generator | None = None,
    ) -> torch.Tensor:
        with autocast(self.device, self.amp_dtype):
            return self.diffusion.sample(
                self.ema, noise, labels, steps=steps, eta=eta,
                guidance_scale=guidance_scale, generator=generator,
            )  # fmt: skip


def build_task(
    model_config: ModelConfig,
    train_config: TrainConfig | None,
    device: torch.device,
    amp_dtype: torch.dtype | None = None,
) -> Task:
    """GANTask or DiffusionTask, from the architecture name."""
    cls = DiffusionTask if model_config.family == "diffusion" else GANTask
    return cls(model_config, train_config, device, amp_dtype)


__all__ = ["DiffusionTask", "GANTask", "Task", "build_task"]


`anim_gen/fid.py`: FID

In [ ]:
%%writefile anim_gen/fid.py
"""Fréchet Inception Distance (FID): how close generated images are to real ones.

Both image sets go through an ImageNet-trained Inception-v3; each set of
2048-d features is summarised by its mean mu and covariance S, and

    FID = ||mu_r - mu_g||^2 + Tr(S_r + S_g - 2 (S_r S_g)^(1/2))

Lower is better; 0 means identical statistics. Note: we use torchvision's
Inception weights, not the TensorFlow ones of the original paper, so numbers
are close to but not exactly comparable with published FIDs. Compare runs of
this repo with each other, at the same number of samples.

    anim-fid --checkpoint checkpoints/run/last.pt --num-samples 10000
"""

from __future__ import annotations

import argparse
import dataclasses
import logging
from collections.abc import Sequence
from pathlib import Path

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset

from anim_gen.runtime import get_dist_info

logger = logging.getLogger(__name__)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class InceptionFeatures(nn.Module):
    """Images in [-1, 1] (any size, 1 or 3 channels) -> [B, 2048] pool features."""

    def __init__(self) -> None:
        super().__init__()
        from torchvision.models import Inception_V3_Weights, inception_v3

        net = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True)
        net.fc = nn.Identity()  # keep the 2048-d pooled features, drop the classifier
        self.net = net.eval().requires_grad_(False)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN)[None, :, None, None])
        self.register_buffer("std", torch.tensor(IMAGENET_STD)[None, :, None, None])

    @torch.no_grad()
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        x = images.float()
        if x.shape[1] == 1:
            x = x.expand(-1, 3, -1, -1)
        x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
        x = ((x + 1) / 2 - self.mean) / self.std
        return self.net(x)


class FeatureStats:
    """Running sum and sum of outer products -> mean and covariance (float64).

    Sums can be added across processes, which makes distributed FID exact.
    """

    def __init__(self, dim: int, device: torch.device | str = "cpu") -> None:
        self.n = torch.zeros((), dtype=torch.float64, device=device)
        self.sum = torch.zeros(dim, dtype=torch.float64, device=device)
        self.outer = torch.zeros(dim, dim, dtype=torch.float64, device=device)

    def add(self, features: torch.Tensor) -> None:
        f = features.to(self.sum.device, torch.float64)
        self.n += f.shape[0]
        self.sum += f.sum(dim=0)
        self.outer += f.T @ f

    def all_reduce(self) -> FeatureStats:
        if get_dist_info().enabled:
            for t in (self.n, self.sum, self.outer):
                dist.all_reduce(t)
        return self

    def mean_cov(self) -> tuple[torch.Tensor, torch.Tensor]:
        n = self.n.clamp(min=2)
        mu = self.sum / n
        cov = (self.outer - n * torch.outer(mu, mu)) / (n - 1)  # unbiased, like numpy.cov
        return mu, cov


def frechet_distance(
    mu1: torch.Tensor, cov1: torch.Tensor, mu2: torch.Tensor, cov2: torch.Tensor
) -> float:
    """FID between two Gaussians.

    Tr((S1 S2)^(1/2)) = sum of the square roots of the eigenvalues of S1 S2
    (they are real and >= 0 in exact arithmetic; we keep the real part).
    """
    mu1, cov1, mu2, cov2 = (t.double().cpu() for t in (mu1, cov1, mu2, cov2))
    eigenvalues = torch.linalg.eigvals(cov1 @ cov2)
    tr_sqrt = eigenvalues.sqrt().real.sum()
    fid = (mu1 - mu2).pow(2).sum() + cov1.trace() + cov2.trace() - 2 * tr_sqrt
    return float(fid.clamp(min=0))


def _shard(n: int) -> range:
    """The slice of 0..n-1 this process handles."""
    info = get_dist_info()
    return range(info.rank, n, info.world_size)


@torch.no_grad()
def dataset_stats(
    dataset: Dataset,
    extractor: nn.Module,
    device: torch.device,
    max_images: int | None = None,
    batch_size: int = 100,
    cache: Path | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Feature mean/cov of (up to `max_images` of) a dataset, cached to `cache`."""
    if cache is not None and Path(cache).exists():
        saved = torch.load(cache, map_location="cpu", weights_only=True)
        return saved["mu"], saved["cov"]
    n = min(len(dataset), max_images or len(dataset))
    loader = DataLoader(Subset(dataset, list(_shard(n))), batch_size=batch_size, num_workers=2)
    stats = FeatureStats(2048, device)
    for images, _ in loader:
        stats.add(extractor(images.to(device)))
    mu, cov = stats.all_reduce().mean_cov()
    mu, cov = mu.cpu(), cov.cpu()
    if cache is not None and get_dist_info().is_main:
        Path(cache).parent.mkdir(parents=True, exist_ok=True)
        torch.save({"mu": mu, "cov": cov, "n": n}, cache)
    return mu, cov


@torch.no_grad()
def generated_stats(
    task,
    extractor: nn.Module,
    num_samples: int,
    batch_size: int = 100,
    seed: int = 0,
    **generate_kwargs,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Feature mean/cov of `num_samples` images from the task's EMA model."""
    info = get_dist_info()
    generator = torch.Generator().manual_seed(seed + info.rank)
    stats = FeatureStats(2048, task.device)
    todo = len(_shard(num_samples))
    while todo > 0:
        n = min(batch_size, todo)
        labels = task.random_labels(n, generator)
        images = task.generate(task.noise(n, generator), labels, **generate_kwargs)
        stats.add(extractor(images))
        todo -= n
    mu, cov = stats.all_reduce().mean_cov()
    return mu.cpu(), cov.cpu()


def main(argv: Sequence[str] | None = None) -> None:
    """CLI: FID of a checkpoint against its training dataset."""
    from anim_gen.data import build_dataset
    from anim_gen.runtime import resolve_device
    from anim_gen.sample import load_task

    p = argparse.ArgumentParser(description="FID of a trained checkpoint.")
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--num-samples", type=int, default=10_000)
    p.add_argument("--batch-size", type=int, default=100)
    p.add_argument("--steps", type=int, default=100, help="DDIM steps (diffusion)")
    p.add_argument("--eta", type=float, default=0.0)
    p.add_argument("--guidance-scale", type=float, default=1.0)
    p.add_argument("--data-dir", type=Path, default=None, help="override the dataset location")
    p.add_argument("--device", default="auto")
    p.add_argument("--seed", type=int, default=0)
    args = p.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    device = resolve_device(args.device)
    task, data_config, _ = load_task(args.checkpoint, device)
    if args.data_dir is not None:
        data_config = dataclasses.replace(data_config, data_dir=args.data_dir)
    dataset = build_dataset(data_config, train=False)  # no flips: compare to real images
    extractor = InceptionFeatures().to(device)
    real = dataset_stats(dataset, extractor, device, max_images=args.num_samples)
    fake = generated_stats(
        task, extractor, args.num_samples, args.batch_size, args.seed,
        steps=args.steps, eta=args.eta, guidance_scale=args.guidance_scale,
    )  # fmt: skip
    fid = frechet_distance(*real, *fake)
    print(f"FID ({args.num_samples} samples): {fid:.2f}")


if __name__ == "__main__":
    main()


__all__ = [
    "FeatureStats",
    "InceptionFeatures",
    "dataset_stats",
    "frechet_distance",
    "generated_stats",
]


`anim_gen/train.py`: the training loop

In [ ]:
%%writefile anim_gen/train.py
"""Train a GAN or a diffusion model.

The file reads top to bottom:

1. data          - DataLoader and an endless stream of batches
2. checkpoints   - save / load everything needed to resume
3. logging       - metrics.jsonl (+ TensorBoard), preview grids
4. train()       - the loop (takes ready-made objects, easy to test)
5. CLI           - main() builds everything from a preset and flags

Run it with:  anim-train --preset mnist-dcgan          (anim-train --help for all flags)
Multi-GPU:    torchrun --nproc_per_node=4 -m anim_gen.train --preset cifar10-resnet
"""

from __future__ import annotations

import argparse
import dataclasses
import json
import logging
import math
import random
import time
from collections.abc import Callable, Iterator, Sequence
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Any

import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler

from anim_gen.config import (
    PRESETS,
    DataConfig,
    ModelConfig,
    TrainConfig,
    config_to_dict,
)
from anim_gen.images import save_grid
from anim_gen.runtime import (
    DistInfo,
    all_reduce_mean,
    get_dist_info,
    local_main_first,
    resolve_device,
    resolve_precision,
    seed_everything,
    setup_distributed,
)
from anim_gen.tasks import Task, build_task

try:  # numpy is optional; we only need it for RNG snapshots
    import numpy as np
except ImportError:  # pragma: no cover
    np = None

logger = logging.getLogger(__name__)

LAST_CHECKPOINT = "last.pt"
BEST_CHECKPOINT = "best.pt"  # lowest FID so far (only written when --fid-every > 0)

# ---------------------------------------------------------------------------
# 1. Data
# ---------------------------------------------------------------------------


def build_loader(dataset: Dataset, config: TrainConfig, device: torch.device) -> DataLoader:
    """Shuffled, full batches only. With several processes each gets its own slice."""
    info = get_dist_info()
    if len(dataset) < config.batch_size * info.world_size:
        raise ValueError(
            f"dataset has {len(dataset)} images, fewer than one batch "
            f"({config.batch_size} x {info.world_size} processes)"
        )
    workers = config.num_workers
    generator = torch.Generator().manual_seed(config.seed + info.rank)
    sampler = (
        DistributedSampler(dataset, shuffle=True, seed=config.seed, drop_last=True)
        if info.enabled
        else None
    )
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=sampler is None,
        sampler=sampler,
        drop_last=True,  # every step sees a full batch
        num_workers=workers,
        pin_memory=device.type == "cuda",
        persistent_workers=workers > 0,
        generator=generator,
    )


def endless(
    loader: DataLoader, start_epoch: int = 0
) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
    """Batches forever, reshuffling at every epoch.

    GANs and diffusion models are trained for a number of *steps*, not epochs,
    and a GAN step may consume several batches (one per discriminator update).
    """
    epoch = start_epoch
    while True:
        if isinstance(loader.sampler, DistributedSampler):
            loader.sampler.set_epoch(epoch)
        yield from loader
        epoch += 1


# ---------------------------------------------------------------------------
# 2. Checkpoints
# ---------------------------------------------------------------------------


@dataclass
class TrainState:
    """Counters that must survive a restart."""

    step: int = 0
    best_fid: float = math.inf
    epoch: int = 0  # passes over the data so far (approximate for GANs, d_steps > 1)


def rng_state() -> dict[str, Any]:
    """Snapshot every RNG so a resumed run continues the same random stream."""
    state: dict[str, Any] = {"python": random.getstate(), "torch": torch.get_rng_state()}
    if np is not None:
        name, keys, pos, has_gauss, cached = np.random.get_state()
        state["numpy"] = (name, keys.tolist(), pos, has_gauss, cached)  # weights_only-safe
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: dict[str, Any]) -> None:
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    if np is not None and "numpy" in state:
        name, keys, pos, has_gauss, cached = state["numpy"]
        np.random.set_state((name, np.array(keys, dtype=np.uint32), pos, has_gauss, cached))
    if torch.cuda.is_available() and "cuda" in state:
        torch.cuda.set_rng_state_all(state["cuda"])


def save_checkpoint(
    path: Path,
    task: Task,
    state: TrainState,
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
    class_names: list[str],
) -> None:
    """Everything needed to resume or to sample. Written atomically."""
    checkpoint = {
        "task": task.state_dict(),
        "step": state.step,
        "epoch": state.epoch,
        "best_fid": state.best_fid,
        "data_config": config_to_dict(data_config),
        "model_config": config_to_dict(model_config),
        "train_config": config_to_dict(train_config),
        "class_names": class_names,
        "rng_state": rng_state(),
    }
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, tmp)
    tmp.replace(path)  # atomic: a crash mid-save never leaves a half-written file


def load_checkpoint(path: Path, task: Task) -> TrainState:
    """Load weights + optimizer states into `task`; return the saved counters."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    task.load_state_dict(checkpoint["task"])
    set_rng_state(checkpoint["rng_state"])
    return TrainState(checkpoint["step"], checkpoint["best_fid"], checkpoint.get("epoch", 0))


# ---------------------------------------------------------------------------
# 3. Logging
# ---------------------------------------------------------------------------


class MetricsLog:
    """Appends one JSON object per event to metrics.jsonl (and TensorBoard if asked).

    A fresh run (start_step 0) starts a new file; a resumed run keeps the
    events up to start_step and drops later ones (written after that checkpoint).
    """

    def __init__(self, output_dir: Path, tensorboard: bool = False, start_step: int = 0) -> None:
        self.path = output_dir / "metrics.jsonl"
        kept = []
        if start_step > 0 and self.path.exists():
            kept = [
                line for line in self.path.read_text().splitlines()
                if line and json.loads(line)["step"] <= start_step
            ]  # fmt: skip
        self.path.write_text("".join(line + "\n" for line in kept))
        self.writer = None
        if tensorboard:
            from torch.utils.tensorboard import SummaryWriter

            self.writer = SummaryWriter(output_dir / "tensorboard")

    def write(self, event: dict[str, Any]) -> None:
        with self.path.open("a") as f:
            f.write(json.dumps(event) + "\n")
        if self.writer is not None:
            for key, value in event.items():
                if isinstance(value, (int, float)) and key not in ("step", "epoch"):
                    self.writer.add_scalar(f"{event['kind']}/{key}", value, event["step"])

    def add_image(self, tag: str, grid_path: Path, step: int) -> None:
        if self.writer is not None:
            import numpy as np  # tensorboard depends on numpy anyway
            from PIL import Image

            image = np.asarray(Image.open(grid_path))
            self.writer.add_image(tag, image, step, dataformats="HWC")

    def close(self) -> None:
        if self.writer is not None:
            self.writer.close()


def preview_inputs(task: Task, n: int, seed: int) -> tuple[torch.Tensor, torch.Tensor | None]:
    """Fixed noise (and labels) for the preview grids, so grids are comparable over time.

    For a conditional model, row r of the grid shows class r % num_classes.
    """
    generator = torch.Generator().manual_seed(seed)
    noise = task.noise(n, generator)
    labels = None
    if task.conditional:
        per_row = max(1, int(math.sqrt(n)))
        labels = (torch.arange(n) // per_row % task.model_config.num_classes).to(task.device)
    return noise, labels


# ---------------------------------------------------------------------------
# 4. Training loop
# ---------------------------------------------------------------------------


def out_of_time_budget(start_time: float, budget_hours: float, device: torch.device) -> bool:
    """True once `budget_hours` have passed. All processes agree (rank 0 decides)."""
    if budget_hours <= 0:
        return False
    flag = torch.tensor(
        float((time.perf_counter() - start_time) / 3600 > budget_hours), device=device
    )
    if get_dist_info().enabled:
        dist.broadcast(flag, src=0)  # clocks differ slightly: never let ranks disagree
    return bool(flag.item())


def train(
    task: Task,
    loader: DataLoader,
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
    class_names: list[str],
    callback: Callable[[dict[str, Any]], None] | None = None,
    fid_fn: Callable[[Task], float] | None = None,
) -> TrainState:
    """Run training and return the final counters.

    Everything is passed in already built, so tests can use tiny models.
    `callback` receives every logged event (main process only); the notebook
    uses it for its live dashboard:
        {"kind": "train", "step", "epoch", "total_steps", "<metric>": value, ..., "it_per_sec"}
        {"kind": "sample", "step", "path"}
        {"kind": "fid", "step", "fid"}
    `fid_fn(task) -> fid` is called every fid_every steps on every process.
    """
    cfg = train_config
    info = get_dist_info()
    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    state = TrainState()
    if cfg.resume is not None:
        state = load_checkpoint(Path(cfg.resume), task)
        logger.info("resumed from %s at step %d", cfg.resume, state.step)

    log = MetricsLog(output_dir, cfg.tensorboard, state.step) if info.is_main else None
    preview_noise, preview_labels = preview_inputs(task, cfg.num_samples, cfg.seed)
    batches_per_epoch = len(loader)
    batches = endless(loader, start_epoch=state.epoch)

    def emit(event: dict[str, Any]) -> None:
        if info.is_main:
            log.write(event)
            if callback is not None:
                callback(event)

    def checkpoint(path: Path) -> None:
        if info.is_main:  # one writer: all processes hold the same weights
            save_checkpoint(
                path, task, state, data_config, model_config, cfg, class_names
            )  # fmt: skip

    def save_preview() -> None:
        if not info.is_main:
            return
        images = task.generate(
            preview_noise, preview_labels, steps=cfg.sample_steps,
            guidance_scale=cfg.guidance_scale,
        )  # fmt: skip
        path = save_grid(images, output_dir / "samples" / f"step_{state.step:07d}.png")
        log.add_image("samples", path, state.step)
        emit({"kind": "sample", "step": state.step, "path": str(path)})

    logger.info(
        "training %s (%s) for %d steps, batch %d x %d processes, on %s, amp=%s",
        model_config.arch, task.family, cfg.steps, cfg.batch_size, info.world_size,
        task.device, task.amp_dtype,
    )  # fmt: skip
    if info.is_main and state.step == 0:
        save_grid(next(iter(loader))[0][: cfg.num_samples], output_dir / "real.png")

    sums: dict[str, torch.Tensor] = {}
    start_time = window_start = time.perf_counter()
    window_steps = 0
    batches_used = 0
    batches_per_step = cfg.d_steps if task.family == "gan" else 1

    while state.step < cfg.steps:
        metrics = task.train_step(batches, state.step)
        state.step += 1
        window_steps += 1
        batches_used += batches_per_step
        if batches_used >= batches_per_epoch:
            state.epoch += 1
            batches_used -= batches_per_epoch
        for key, value in metrics.items():  # stays on device: no GPU sync every step
            sums[key] = sums.get(key, 0.0) + value

        if state.step % cfg.log_every == 0 or state.step == cfg.steps:
            elapsed = time.perf_counter() - window_start
            means = {k: float(v) / window_steps for k, v in sums.items()}
            means = all_reduce_mean(means, task.device)
            if not all(math.isfinite(v) for v in means.values()):
                raise FloatingPointError(f"non-finite metrics at step {state.step}: {means}")
            emit({
                "kind": "train", "step": state.step, "epoch": state.epoch,
                "total_steps": cfg.steps, **means, "it_per_sec": window_steps / elapsed,
            })  # fmt: skip
            logger.info(
                "step %d/%d | %s | %.1f it/s",
                state.step, cfg.steps, " | ".join(f"{k} {v:.4f}" for k, v in means.items()),
                window_steps / elapsed,
            )  # fmt: skip
            sums, window_steps = {}, 0
            window_start = time.perf_counter()

        if state.step % cfg.sample_every == 0 or state.step == cfg.steps:
            save_preview()

        if (
            fid_fn is not None
            and cfg.fid_every > 0
            and (state.step % cfg.fid_every == 0 or state.step == cfg.steps)
        ):
            fid = fid_fn(task)
            emit({"kind": "fid", "step": state.step, "fid": fid})
            logger.info("fid | step %d | %.2f", state.step, fid)
            if fid < state.best_fid:
                state.best_fid = fid
                checkpoint(output_dir / BEST_CHECKPOINT)

        out_of_time = state.step % cfg.log_every == 0 and out_of_time_budget(
            start_time, cfg.time_budget_hours, task.device
        )
        if state.step % cfg.ckpt_every == 0 or state.step == cfg.steps or out_of_time:
            checkpoint(output_dir / LAST_CHECKPOINT)
        if out_of_time:
            logger.info("time budget of %.2f h used: stopping at step %d", cfg.time_budget_hours,
                        state.step)  # fmt: skip
            break
        if state.step % cfg.log_every == 0:
            window_start = time.perf_counter()  # don't count sampling/eval time in it/s

    if log is not None:
        log.close()
    logger.info("done | %d steps | outputs in %s", state.step, output_dir)
    return state


# ---------------------------------------------------------------------------
# 5. Command line
# ---------------------------------------------------------------------------

# Fields the CLI does not expose: they are read from the dataset.
DERIVED_MODEL_FIELDS = {"image_size", "channels", "num_classes"}


def _add_field(group: argparse._ArgumentGroup, f: dataclasses.Field, default: Any) -> None:
    """One --flag per config field; the type is taken from the default value."""
    flag = "--" + f.name.replace("_", "-")
    if isinstance(default, bool):
        group.add_argument(flag, action=argparse.BooleanOptionalAction, default=default)
    elif isinstance(default, tuple):
        group.add_argument(flag, type=int, nargs="+", default=default)
    elif isinstance(default, Path) or f.name in ("data_dir", "output_dir", "resume"):
        group.add_argument(flag, type=Path, default=default)
    elif f.name == "limit":
        group.add_argument(flag, type=int, default=default)
    else:
        group.add_argument(flag, type=type(default), default=default)


def build_parser() -> argparse.ArgumentParser:
    """All CLI flags; defaults come from the config dataclasses (then the preset)."""
    p = argparse.ArgumentParser(
        description="Train a GAN (dcgan, resnet) or a diffusion model (unet).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--preset", choices=sorted(PRESETS), default=None, help="starting config")
    p.add_argument("--list-presets", action="store_true", help="print the presets and exit")
    for title, cls in (("data", DataConfig), ("model", ModelConfig), ("training", TrainConfig)):
        group = p.add_argument_group(title)
        defaults = cls()
        for f in fields(cls):
            if cls is ModelConfig and f.name in DERIVED_MODEL_FIELDS:
                continue
            _add_field(group, f, getattr(defaults, f.name))
    return p


def parse_args(argv: Sequence[str] | None = None) -> argparse.Namespace:
    """Parse twice: first find --preset, then use it as the defaults."""
    parser = build_parser()
    known, _ = parser.parse_known_args(argv)
    if known.preset is not None:
        parser.set_defaults(**PRESETS[known.preset])
    return parser.parse_args(argv)


def configs_from_args(args: argparse.Namespace) -> tuple[DataConfig, ModelConfig, TrainConfig]:
    """Parsed flags -> the three configs. num_classes is filled in once the data is loaded."""

    def pick(cls: type) -> dict[str, Any]:
        names = {f.name for f in fields(cls)}
        if cls is ModelConfig:
            names -= DERIVED_MODEL_FIELDS
        return {
            k: (tuple(v) if isinstance(v, list) else v) for k, v in vars(args).items() if k in names
        }

    data = DataConfig(**pick(DataConfig))
    model = ModelConfig(**pick(ModelConfig), image_size=data.image_size, channels=data.channels)
    return data, model, TrainConfig(**pick(TrainConfig))


def main(argv: Sequence[str] | None = None) -> None:
    """CLI entry point. Returns None (console scripts pass the return to sys.exit)."""
    args = parse_args(argv)
    if args.list_presets:
        for name, preset in PRESETS.items():
            print(f"{name:20s} arch={preset['arch']:7s} dataset={preset['dataset']}")
        return
    data_config, model_config, train_config = configs_from_args(args)
    info = setup_distributed(train_config.device)
    logging.basicConfig(
        level=logging.INFO if info.is_main else logging.WARNING,
        format="%(asctime)s %(levelname)s %(message)s",
        datefmt="%H:%M:%S",
    )
    try:
        build_and_train(data_config, model_config, train_config, info)
    finally:
        if info.enabled:
            dist.destroy_process_group()


def build_and_train(
    data_config: DataConfig,
    model_config: ModelConfig,
    train_config: TrainConfig,
    info: DistInfo | None = None,
    callback: Callable[[dict[str, Any]], None] | None = None,
) -> tuple[Task, TrainState]:
    """Dataset -> model -> train. Shared by the CLI and the notebook."""
    from anim_gen.data import build_dataset, dataset_classes, num_classes
    from anim_gen.models import count_parameters

    info = info or get_dist_info()
    seed_everything(train_config.seed)  # same initial weights on every run
    with local_main_first(info):  # one download per machine
        dataset = build_dataset(data_config, train=True)
    class_names = dataset_classes(dataset)
    model_config = dataclasses.replace(
        model_config,
        image_size=data_config.image_size,
        channels=data_config.channels,
        num_classes=num_classes(data_config, dataset),
    )
    device = resolve_device(train_config.device)
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True  # free speed-up on Ampere+, harmless elsewhere
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True  # fixed input sizes: let cuDNN pick fast kernels
    amp_dtype = resolve_precision(train_config.precision, device)
    task = build_task(model_config, train_config, device, amp_dtype)
    loader = build_loader(dataset, train_config, device)

    sizes = ", ".join(
        f"{name} {count_parameters(net) / 1e6:.2f}M"
        for name, net in task.networks.items()
        if name not in ("G_ema", "ema")
    )
    logger.info(
        "data: %s, %d images, %d classes%s | model: %s",
        data_config.dataset, len(dataset), len(class_names),
        " (conditional)" if model_config.num_classes else "", sizes,
    )  # fmt: skip

    fid_fn = None
    if train_config.fid_every > 0:
        fid_fn = make_fid_fn(data_config, train_config, device)
    state = train(
        task, loader, data_config, model_config, train_config, class_names,
        callback=callback, fid_fn=fid_fn,
    )  # fmt: skip
    return task, state


def make_fid_fn(
    data_config: DataConfig, train_config: TrainConfig, device: torch.device
) -> Callable[[Task], float]:
    """FID of the task's EMA model against the (cached) real-data statistics."""
    from anim_gen.data import build_dataset
    from anim_gen.fid import InceptionFeatures, dataset_stats, frechet_distance, generated_stats

    extractor = InceptionFeatures().to(device)
    n = train_config.fid_samples
    cache = Path(train_config.output_dir) / f"fid_stats_real_{n}.pt"
    real = dataset_stats(build_dataset(data_config, train=False), extractor, device, n, cache=cache)

    def fid_fn(task: Task) -> float:
        fake = generated_stats(
            task, extractor, n, seed=train_config.seed,
            steps=train_config.sample_steps, guidance_scale=train_config.guidance_scale,
        )  # fmt: skip
        return frechet_distance(*real, *fake)

    return fid_fn


if __name__ == "__main__":
    main()


`anim_gen/sample.py`: sampling helpers

In [ ]:
%%writefile anim_gen/sample.py
"""Generate images from a trained checkpoint.

    anim-sample --checkpoint checkpoints/run/last.pt --num 64 --out samples.png
    anim-sample --checkpoint ... --class dog --num 16 --out dogs.png
    anim-sample --checkpoint ... --num 1000 --folder generated/      # one PNG per image
    anim-sample --checkpoint ... --interpolate 8 --out walk.png      # latent walks

Sampling always uses the EMA weights stored in the checkpoint.
"""

from __future__ import annotations

import argparse
import logging
import math
from collections.abc import Sequence
from pathlib import Path

import torch

from anim_gen.config import DataConfig, ModelConfig, config_from_dict
from anim_gen.images import save_grid, save_images
from anim_gen.runtime import resolve_device, resolve_precision
from anim_gen.tasks import Task, build_task

logger = logging.getLogger(__name__)


def load_task(
    path: Path, device: torch.device | str = "cpu", precision: str = "fp32"
) -> tuple[Task, DataConfig, list[str]]:
    """Rebuild the model from the configs saved in a checkpoint and load its weights.

    Returns (task, data_config, class_names). The task has no optimizers.
    """
    device = torch.device(device)
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model_config = config_from_dict(ModelConfig, checkpoint["model_config"])
    data_config = config_from_dict(DataConfig, checkpoint["data_config"])
    task = build_task(model_config, None, device, resolve_precision(precision, device))
    task.load_state_dict(checkpoint["task"])
    return task, data_config, list(checkpoint["class_names"])


def slerp(a: torch.Tensor, b: torch.Tensor, t: float) -> torch.Tensor:
    """Spherical interpolation: stays on the shell where Gaussian noise lives.

    A straight line between two noise vectors passes through low-norm points
    the model never saw in training; slerp keeps the norm about constant.
    """
    a_flat, b_flat = a.flatten(1), b.flatten(1)
    cos = (a_flat * b_flat).sum(1) / (a_flat.norm(dim=1) * b_flat.norm(dim=1))
    omega = torch.acos(cos.clamp(-1 + 1e-7, 1 - 1e-7))[:, None]
    so = torch.sin(omega)
    mixed = (torch.sin((1 - t) * omega) / so) * a_flat + (torch.sin(t * omega) / so) * b_flat
    return mixed.view_as(a)


def resolve_labels(
    task: Task, class_names: list[str], name: str | None, n: int, generator: torch.Generator
) -> torch.Tensor | None:
    """Labels for n images: the named class, or random classes, or None."""
    if not task.conditional:
        if name is not None:
            raise ValueError("this model is unconditional: --class is not available")
        return None
    if name is None:
        return task.random_labels(n, generator)
    if name.isdigit():
        index = int(name)
    elif name in class_names:
        index = class_names.index(name)
    else:
        raise ValueError(f"unknown class {name!r}; choose from {class_names}")
    return torch.full((n,), index, device=task.device, dtype=torch.long)


@torch.no_grad()
def generate(
    task: Task,
    num: int,
    labels: torch.Tensor | None = None,
    batch_size: int = 64,
    generator: torch.Generator | None = None,
    **kwargs,
) -> torch.Tensor:
    """`num` images in [-1, 1], generated in batches, returned on the CPU."""
    out = []
    for start in range(0, num, batch_size):
        n = min(batch_size, num - start)
        batch_labels = None if labels is None else labels[start : start + n]
        out.append(task.generate(task.noise(n, generator), batch_labels, **kwargs).cpu())
    return torch.cat(out)


@torch.no_grad()
def interpolate(
    task: Task, rows: int, cols: int = 8, labels: torch.Tensor | None = None,
    generator: torch.Generator | None = None, **kwargs,
) -> torch.Tensor:  # fmt: skip
    """`rows` walks of `cols` images, each between two random noise inputs."""
    start, end = task.noise(rows, generator), task.noise(rows, generator)
    frames = [slerp(start, end, t) for t in torch.linspace(0, 1, cols).tolist()]
    noise = torch.stack(frames, dim=1).flatten(0, 1)  # row-major: [rows * cols, ...]
    if labels is not None:
        labels = labels[:rows].repeat_interleave(cols)
    return task.generate(noise, labels, **kwargs).cpu()


def main(argv: Sequence[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Generate images from a trained checkpoint.")
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--num", type=int, default=64, help="number of images")
    p.add_argument("--out", type=Path, default=Path("samples.png"), help="grid PNG")
    p.add_argument("--folder", type=Path, default=None, help="also save every image here")
    p.add_argument("--class", dest="class_name", default=None, help="class name or id")
    p.add_argument("--interpolate", type=int, metavar="ROWS", default=0, help="latent walks")
    p.add_argument("--steps", type=int, default=100, help="DDIM steps (diffusion only)")
    p.add_argument("--eta", type=float, default=0.0, help="0 = DDIM, 1 = DDPM-like noise")
    p.add_argument("--guidance-scale", type=float, default=1.0, help="classifier-free guidance")
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="auto")
    p.add_argument("--precision", default="fp32", choices=["auto", "fp32", "bf16"])
    args = p.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    device = resolve_device(args.device)
    task, _, class_names = load_task(args.checkpoint, device, args.precision)
    generator = torch.Generator().manual_seed(args.seed)
    kwargs = {"steps": args.steps, "eta": args.eta, "guidance_scale": args.guidance_scale}

    if args.interpolate:
        labels = resolve_labels(task, class_names, args.class_name, args.interpolate, generator)
        images = interpolate(task, args.interpolate, 8, labels, generator, **kwargs)
        save_grid(images, args.out, nrow=8)
    else:
        labels = resolve_labels(task, class_names, args.class_name, args.num, generator)
        images = generate(task, args.num, labels, args.batch_size, generator, **kwargs)
        save_grid(images, args.out, nrow=max(1, int(math.sqrt(args.num))))
        if args.folder is not None:
            save_images(images, args.folder)
            logger.info("wrote %d images to %s", len(images), args.folder)
    logger.info("wrote %s", args.out)


if __name__ == "__main__":
    main()


__all__ = ["generate", "interpolate", "load_task", "slerp"]


`anim_gen/prepare.py`: dataset preparation (AFHQ)

In [ ]:
%%writefile anim_gen/prepare.py
"""Build a training image folder from a Hugging Face dataset, a folder or a zip.

High-resolution datasets are slow to decode and resize at every step, so we
do it once: centre-crop to a square, resize with a high-quality filter, and
write one PNG per image, one sub-folder per class:

    data/afhq64/cat/000000.png  data/afhq64/dog/000001.png  ...

Examples:
    # AFHQ (Animal Faces HQ: cat / dog / wild, 512x512), streamed from the Hub
    anim-prepare --source hf:huggan/AFHQ --sizes 64 128 --out 'data/afhq{size}'
    # your own photos: a folder (sub-folders = classes) or a zip of one
    anim-prepare --source ~/Pictures/animals.zip --sizes 64 --out 'data/animals{size}'

Then train with:  anim-train --preset afhq64-resnet --data-dir data/afhq64
"""

from __future__ import annotations

import argparse
import io
import logging
import zipfile
from collections.abc import Iterator, Sequence
from pathlib import Path, PurePosixPath

from PIL import Image

from anim_gen.data import IMAGE_EXTENSIONS

logger = logging.getLogger(__name__)


def center_square(image: Image.Image) -> Image.Image:
    """Crop the largest centred square."""
    w, h = image.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    return image.crop((left, top, left + side, top + side))


def resize(image: Image.Image, size: int) -> Image.Image:
    """Square-crop and resize with Lanczos (sharp, little aliasing)."""
    return center_square(image.convert("RGB")).resize((size, size), Image.Resampling.LANCZOS)


def iter_folder(root: Path) -> Iterator[tuple[Image.Image, str]]:
    """(image, class name) for every image under root; sub-folders are classes."""
    subdirs = sorted(p for p in root.iterdir() if p.is_dir())
    groups = [(d, d.name) for d in subdirs] or [(root, "all")]
    for folder, label in groups:
        for path in sorted(folder.rglob("*")):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                with Image.open(path) as img:
                    yield img.convert("RGB"), label


def iter_zip(path: Path) -> Iterator[tuple[Image.Image, str]]:
    """Same as iter_folder, reading straight from a zip archive (no extraction)."""
    with zipfile.ZipFile(path) as archive:
        names = sorted(
            n for n in archive.namelist()
            if PurePosixPath(n).suffix.lower() in IMAGE_EXTENSIONS and "__MACOSX" not in n
        )  # fmt: skip
        parents = {PurePosixPath(n).parent for n in names}
        # One common parent = flat archive (one class); otherwise the parent folder is the class.
        flat = len(parents) == 1
        for name in names:
            label = "all" if flat else PurePosixPath(name).parent.name
            with Image.open(io.BytesIO(archive.read(name))) as img:
                yield img.convert("RGB"), label


def iter_hf(
    dataset_id: str, split: str, image_column: str, label_column: str | None
) -> Iterator[tuple[Image.Image, str]]:
    """Stream a Hugging Face image dataset (nothing is cached on disk)."""
    try:
        from datasets import load_dataset
    except ImportError as e:  # pragma: no cover
        raise SystemExit("anim-prepare hf:... needs `pip install -e '.[hf]'`") from e
    dataset = load_dataset(dataset_id, split=split, streaming=True)
    names = None
    if label_column is not None:
        feature = dataset.features.get(label_column) if dataset.features else None
        names = getattr(feature, "names", None)
    for row in dataset:
        label = "all"
        if label_column is not None and label_column in row:
            value = row[label_column]
            label = names[value] if names is not None else str(value)
        yield row[image_column].convert("RGB"), label


def prepare(
    images: Iterator[tuple[Image.Image, str]],
    out_pattern: str,
    sizes: Sequence[int],
    limit: int | None = None,
) -> dict[int, int]:
    """Write every image at every size. Returns {size: images written}."""
    counts = dict.fromkeys(sizes, 0)
    for index, (image, label) in enumerate(images):
        if limit is not None and index >= limit:
            break
        for size in sizes:
            folder = Path(out_pattern.format(size=size)) / label
            folder.mkdir(parents=True, exist_ok=True)
            resize(image, size).save(folder / f"{index:06d}.png")
            counts[size] += 1
        if (index + 1) % 1000 == 0:
            logger.info("%d images done", index + 1)
    return counts


def main(argv: Sequence[str] | None = None) -> None:
    p = argparse.ArgumentParser(description="Resize a dataset into a training image folder.")
    p.add_argument("--source", required=True, help="hf:<dataset id>, a folder, or a .zip")
    p.add_argument("--out", required=True, help="output folder; '{size}' is replaced per size")
    p.add_argument("--sizes", type=int, nargs="+", default=[64])
    p.add_argument("--split", default="train", help="Hugging Face split")
    p.add_argument("--image-column", default="image")
    p.add_argument("--label-column", default="label", help="'' for no classes")
    p.add_argument("--limit", type=int, default=None, help="stop after N images")
    args = p.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)  # hide Hugging Face HTTP logs

    if len(args.sizes) > 1 and "{size}" not in args.out:
        p.error("with several --sizes, --out must contain '{size}'")
    if args.source.startswith("hf:"):
        images = iter_hf(args.source[3:], args.split, args.image_column, args.label_column or None)
    else:
        source = Path(args.source).expanduser()
        images = iter_zip(source) if source.suffix == ".zip" else iter_folder(source)
    counts = prepare(images, args.out, args.sizes, args.limit)
    for size, n in counts.items():
        logger.info("%s: %d images", args.out.format(size=size), n)


if __name__ == "__main__":
    main()


__all__ = ["center_square", "iter_folder", "iter_hf", "iter_zip", "prepare", "resize"]


`anim_gen/__init__.py`: package

In [ ]:
%%writefile anim_gen/__init__.py
"""anim-gen: image generation with GANs (DCGAN, ResNet-SNGAN) and diffusion (DDPM/DDIM U-Net)."""

from anim_gen.config import PRESETS, DataConfig, ModelConfig, TrainConfig

__all__ = ["PRESETS", "DataConfig", "ModelConfig", "TrainConfig"]
__version__ = "0.1.0"


In [ ]:
import importlib
import sys

sys.path.insert(0, os.getcwd())
importlib.invalidate_caches()
import anim_gen

print("anim_gen", anim_gen.__version__, "ready")

## 3. Settings

Pick a preset (or change any field below it). Good first choices:

| preset | what | T4 time (rough) |
|---|---|---|
| `mnist-dcgan` | classic DCGAN on digits | minutes |
| `cifar10-resnet` | class-conditional ResNet GAN, hinge loss, spectral norm, self-attention | a few hours |
| `cifar10-diffusion` | class-conditional DDPM U-Net, 38M params | many hours (resume across sessions) |
| `afhq64-resnet` | animal faces 64×64, ResNet GAN + DiffAugment | several hours |
| `afhq64-diffusion` | animal faces 64×64, U-Net, v-prediction | long (resume across sessions) |

`SMOKE_TEST = True` runs a tiny version of the chosen preset in a few minutes,
to check the whole notebook. Keep it `False` for real training.

In [ ]:
import dataclasses
import logging
from pathlib import Path

import torch

from anim_gen.train import configs_from_args, parse_args

PRESET = "cifar10-resnet"   # see the table above, or anim_gen.config.PRESETS
SMOKE_TEST = False          # True = tiny quick check of the whole notebook
USE_GOOGLE_DRIVE = False    # Colab only: keep checkpoints on Drive (survive disconnects)
RESUME = True               # continue from last.pt in OUTPUT_DIR if it exists
TIME_BUDGET_HOURS = 11.0    # stop cleanly (and save) before the 12 h session limit
FID_EVERY = 0               # e.g. 5000: FID on 5k samples (downloads Inception, ~100 MB)
OVERRIDES = {}              # any config field, e.g. {"steps": 20000, "batch_size": 128}

# --- where things go -------------------------------------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ
if ON_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/anim-gen")
elif ON_KAGGLE:
    WORK = Path("/kaggle/working")
elif ON_COLAB:
    WORK = Path("/content")
else:
    WORK = Path(".")
DATA_DIR = (Path("/kaggle/temp") if ON_KAGGLE else WORK) / "data"  # Kaggle: temp = not saved
OUTPUT_DIR = WORK / "checkpoints" / (PRESET + ("-smoke" if SMOKE_TEST else ""))

# --- configs: preset -> overrides -> environment ---------------------------------------
args = parse_args(["--preset", PRESET])
data_config, model_config, train_config = configs_from_args(args)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = DATA_DIR / data_config.data_dir.name if data_config.dataset == "folder" else DATA_DIR
env = {
    "data_dir": data_dir, "output_dir": OUTPUT_DIR, "device": device.type,
    "num_workers": min(4, os.cpu_count() or 1), "fid_every": FID_EVERY,
    "time_budget_hours": TIME_BUDGET_HOURS, "log_every": 100,
}
if SMOKE_TEST:
    env.update(steps=200, log_every=20, sample_every=100, ckpt_every=100, limit=512,
               batch_size=32, fid_every=0, sample_steps=20)
settings = {**env, **OVERRIDES}


def apply(config, values):
    names = {f.name for f in dataclasses.fields(config)}
    return dataclasses.replace(config, **{k: v for k, v in values.items() if k in names})


data_config = apply(data_config, settings)
model_config = apply(model_config, settings)
train_config = apply(train_config, settings)
if RESUME and (OUTPUT_DIR / "last.pt").exists():
    train_config = dataclasses.replace(train_config, resume=OUTPUT_DIR / "last.pt")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.0f} GB), compute {props.major}.{props.minor}"
          f" -> precision {'bf16' if props.major >= 8 else 'fp32'}")
else:
    print("No GPU: enable one in the notebook settings (fine only for SMOKE_TEST).")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S",
                    force=True)
for name, cfg in [("data", data_config), ("model", model_config), ("train", train_config)]:
    print(f"\n{name}:", ", ".join(f"{k}={v}" for k, v in dataclasses.asdict(cfg).items()))

## 4. Data

MNIST and CIFAR-10 download automatically. AFHQ (Animal Faces HQ, 512×512,
~15k images of cats, dogs and wild animals) is streamed from the Hugging Face
Hub and resized once to the preset's resolution (~10 minutes for 64×64).
If the Hub rate-limits you, add a token: *Add-ons → Secrets* `HF_TOKEN` on
Kaggle, or `os.environ["HF_TOKEN"] = "..."`.

In [ ]:
from anim_gen.data import build_dataset, dataset_classes
from anim_gen.prepare import iter_hf, prepare

if data_config.dataset == "folder" and not any(data_config.data_dir.rglob("*.png")):
    print(f"preparing AFHQ at {data_config.image_size}px in {data_config.data_dir} ...")
    limit = data_config.limit if SMOKE_TEST else None
    counts = prepare(iter_hf("huggan/AFHQ", "train", "image", "label"),
                     str(data_config.data_dir), [data_config.image_size], limit=limit)
    print(counts)

dataset = build_dataset(data_config)
print(f"{len(dataset):,} images, classes: {dataset_classes(dataset)}")

In [ ]:
import matplotlib.pyplot as plt

from anim_gen.images import image_grid


def show(images, title="", nrow=None, size=6):
    grid = image_grid(images, nrow=nrow)
    plt.figure(figsize=(size, size))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.title(title)
    plt.axis("off")
    plt.show()


real = torch.stack([dataset[i][0] for i in torch.randperm(len(dataset))[:64].tolist()])
show(real, "real training images")

## 5. Model

`build_and_train` (next sections) builds the networks from the configs; here
we build them once on their own to show their sizes.

In [ ]:
from anim_gen.data import num_classes
from anim_gen.models import count_parameters
from anim_gen.tasks import build_task

shaped = dataclasses.replace(model_config, num_classes=num_classes(data_config, dataset))
preview_task = build_task(shaped, train_config, torch.device("cpu"))
for name, net in preview_task.networks.items():
    if name not in ("G_ema", "ema"):
        print(f"{name:5} {count_parameters(net) / 1e6:6.2f}M parameters")
print("family:", shaped.family, "| conditional:", shaped.num_classes > 0)
del preview_task

## 6. Live dashboard

Redrawn at every log step: the losses, and the newest preview grid (same
fixed noise every time, so you can watch the samples sharpen; for a
conditional model each row is one class).

- **GAN**: `d_loss` / `g_loss` should *not* go to 0; a stable run keeps them
  roughly level. `d_real` / `d_fake` are the discriminator's mean logits.
- **diffusion**: `loss` is the denoising MSE (starts ≈ 1, falls fast, then slowly).

In [ ]:
import json

from IPython.display import clear_output
from PIL import Image

history = {"train": [], "fid": [], "sample": None}
if train_config.resume is not None and (OUTPUT_DIR / "metrics.jsonl").exists():
    for line in (OUTPUT_DIR / "metrics.jsonl").read_text().splitlines():
        event = json.loads(line)
        if event["kind"] == "sample":
            history["sample"] = event["path"]
        elif event["kind"] in history:
            history[event["kind"]].append(event)


def dashboard(event):
    kind = event["kind"]
    if kind == "sample":
        history["sample"] = event["path"]
        return
    history[kind].append(event)
    if kind != "train":
        return
    rows = history["train"]
    skip = {"kind", "step", "epoch", "total_steps", "it_per_sec"}
    keys = [k for k in rows[-1] if k not in skip]
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [3, 2]})
    steps = [r["step"] for r in rows]
    for key in keys:
        ax1.plot(steps, [r.get(key, float("nan")) for r in rows], label=key)
    for r in history["fid"]:
        ax1.annotate(f"FID {r['fid']:.1f}", (r["step"], 0), fontsize=8)
    ax1.set_xlabel("step")
    ax1.grid(alpha=0.3)
    ax1.legend()
    last = rows[-1]
    eta = (last["total_steps"] - last["step"]) / max(last["it_per_sec"], 1e-9) / 60
    ax1.set_title(f"step {last['step']:,}/{last['total_steps']:,} | "
                  f"{last['it_per_sec']:.1f} it/s | ETA {eta:.0f} min")
    if history["sample"]:
        ax2.imshow(Image.open(history["sample"]))
        ax2.set_title(Path(history["sample"]).stem)
    ax2.axis("off")
    fig.tight_layout()
    plt.show()

## 7. Training

Same code as the repo's `anim-train`. It writes to `OUTPUT_DIR`:
`last.pt` (every `ckpt_every` steps), `best.pt` (lowest FID, if `FID_EVERY > 0`),
`samples/step_*.png`, `real.png` and `metrics.jsonl`. The checkpoints work with
the repo's `anim-sample`, `anim-fid` and `anim-train --resume`.

**Interrupted, or hit the time budget?** Run the notebook again with
`RESUME = True`: it continues from `last.pt`. On Kaggle, to continue in a *new*
session, add the previous version's output as input data and copy its
`checkpoints/` folder to `/kaggle/working` first.

In [ ]:
from anim_gen.train import build_and_train

task, state = build_and_train(data_config, model_config, train_config, callback=dashboard)
print(f"stopped at step {state.step:,}; outputs in {OUTPUT_DIR}")

## 8. Results

Samples from the EMA weights. For a conditional model: one row per class, then
latent walks (spherical interpolation between two noise inputs).

In [ ]:
from anim_gen.sample import generate, interpolate

g = torch.Generator().manual_seed(0)
kwargs = {"steps": 100, "guidance_scale": 2.0 if task.conditional else 1.0}
if task.conditional:
    classes = dataset_classes(dataset)
    per_class = 8
    labels = torch.arange(len(classes)).repeat_interleave(per_class).to(task.device)
    images = generate(task, len(labels), labels, generator=g, **kwargs)
    show(images, "one row per class: " + ", ".join(classes), nrow=per_class, size=8)
else:
    show(generate(task, 64, generator=g, **kwargs), "samples", size=8)
walk = interpolate(task, rows=4, cols=8, labels=task.random_labels(4, g), generator=g, **kwargs)
show(walk, "latent walks", nrow=8, size=8)

## 9. Download the trained model

`anim-gen-run.zip` contains `last.pt`, `best.pt` (if any), the sample grids
and the metrics. Locally: `anim-sample --checkpoint last.pt --num 64`.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK / "anim-gen-run"), "zip", OUTPUT_DIR)
print("saved", archive)
if ON_COLAB:
    from google.colab import files

    files.download(archive)